# Стаття 1. Відтворюваний аудит генерації українського рукописного тексту

Увесь код аудиту: кожен файл проєкту записано окремою клітинкою `%%writefile` (та сама версія, що в репозиторії). Код методів, ваги й шрифти скрипти завантажують самі на зафіксованих ревізіях (`versions/pinned.json`).

| Крок | Що | Де |
|---|---|---|
| 1 | Інвентар символів (131 запис) | CPU |
| 2 | Фіксація версій: 14 git-репозиторіїв, 11 ревізій Hugging Face, Kobzar KS (23.10.2026) | CPU |
| 3 | Рівні 1–2 для 11 реалізацій методів | CPU |
| 4 | Навчальні дані checkpoint | вручну, `checkpoints/` |
| 5 | Аудит 59 шрифтів | CPU |
| 6 | Кодування форм 9 літер двома кодувальниками | вручну, `coding_kit/` |
| 7 | Рівень 3: генерація інвентарю 5 моделями, сліпа оцінка | **GPU (T4)** |
| 8 | Матриця вимог і рекомендації | CPU, `scripts/build_matrix.py` |

**Як користуватися.** Розділ «Налаштування» виконується після кожного нового сеансу. Крок 7 пише результати на Google Drive (`MyDrive/ukr-htg-audit/level3`), тож після обриву сеансу досить повторити «Налаштування» і запуск — генерація продовжиться з місця зупинки. Довгі запуски йдуть у фоні (`subprocess.Popen`); стан показує клітинка «Стан генерації».

## Налаштування

GPU, Google Drive, тека проєкту `/content/ukr-htg-audit`, змінні середовища й залежності.

In [ ]:
# GPU (для кроку 7 потрібен; Середовище виконання → Змінити тип → T4 GPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv; python3 --version

In [ ]:
# Google Drive (результати кроку 7), тека проєкту, змінні середовища
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/ukr-htg-audit'
PROJECT = '/content/ukr-htg-audit'
os.makedirs(DRIVE_DIR, exist_ok=True)
for d in ['inventory', 'versions', 'scripts/level12', 'scripts/fonts', 'scripts/level3', 'scripts/coding', 'checkpoints', 'level3']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)
os.environ['AUDIT_WORK'] = '/content/work'           # код методів, ваги, шрифти — локальний диск
os.environ['HF_HOME'] = '/content/hf'
os.environ['HF_HUB_DISABLE_XET'] = '1'                # xet-завантаження іноді обривається
os.environ['LEVEL3_OUT'] = f'{DRIVE_DIR}/level3'      # результати кроку 7 — на Drive
%cd /content/ukr-htg-audit

In [ ]:
%%writefile requirements-audit.txt
# Середовище аудиту (локально на CPU і в Colab). torch/torchvision у Colab вже є.
torch
torchvision
transformers
diffusers
accelerate
huggingface_hub
sentencepiece
timm
einops
omegaconf
numpy
pillow
opencv-python-headless
scikit-image
scipy
pandas
h5py
matplotlib
tqdm
lmdb
msgpack
wandb
gdown
fonttools
uharfbuzz

In [ ]:
# Залежності (torch і torchvision у Colab уже встановлено).
# torchao у Colab несумісний із новим diffusers (ImportError FqnToConfig) і моделям аудиту не потрібен — прибираємо.
!grep -v '^torch' requirements-audit.txt | grep -v '^#' > /tmp/req.txt && pip -q install -r /tmp/req.txt
!pip -q uninstall -y torchao

In [ ]:
# Окремі середовища для Emuru й Eruku: їхній віддалений код (trust_remote_code) не працює з transformers 5.x.
# Версії — з requirements.txt репозиторіїв (Emuru d8db91a, Eruku 3bcddbd); torch — системний з Colab.
# Встановлення йде у фоні (~1–2 хв), журнал — /content/venvs.log.
import subprocess
VENVS = {
    "emuru": "transformers==4.54.0 tokenizers==0.21.4 huggingface-hub==0.34.2 diffusers==0.34.0 accelerate==1.9.0",
    "eruku": "transformers==4.53.0 huggingface-hub==0.34.2 diffusers==0.34.0 accelerate==1.8.1",
}
PY = {m: f"/content/venv_{m}/bin/python" for m in VENVS}   # інші моделі — системний python3
script = " && ".join(f"python3 -m venv --system-site-packages --without-pip /content/venv_{m} && "
                     f"/content/venv_{m}/bin/python -m pip -q install {pins} && echo '{m}: готово'"
                     for m, pins in VENVS.items())
venvs = subprocess.Popen(["bash", "-c", script], stdout=open("/content/venvs.log", "w"), stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL, start_new_session=True)
print("встановлення запущено, pid", venvs.pid)

## Крок 1. Інвентар символів

`inventory/charset_inventory.json` — єдине джерело символів для всіх скриптів: 33 літери в обох регістрах, апостроф у трьох кодуваннях, розділові знаки правопису 2019 р., історичні знаки XIX — поч. XX ст. (звірено з хрестоматією Німчука 2004 р.).

In [ ]:
%%writefile inventory/build_inventory.py
"""Будує єдиний інвентар символів для аудиту (стаття 1).

Результат: inventory/charset_inventory.json — його читають усі скрипти аудиту.
Запуск: python3 inventory/build_inventory.py
"""

import json
import sys
import unicodedata
from datetime import date
from pathlib import Path

OUT = Path(__file__).with_name("charset_inventory.json")
INVENTORY_VERSION = "0.3.0"

# --- 1. Алфавіт ---------------------------------------------------------------
# 33 літери сучасної абетки (з ґ). Окремого параграфа з абеткою правопис 2019 р.
# не має; ґ згадано в передмові як відновлену.
ALPHABET_LOWER = [
    "а", "б", "в", "г", "ґ", "д", "е", "є", "ж", "з", "и", "і", "ї", "й",
    "к", "л", "м", "н", "о", "п", "р", "с", "т", "у", "ф", "х", "ц", "ч",
    "ш", "щ", "ь", "ю", "я",
]

# Літери, рукописна форма яких (за прописами) відрізняється від друкованої.
# Позначено мінускули: саме їх кодуємо за прописами на рівні 4.
KEY_HANDWRITTEN = ["т", "д", "и", "п", "ш", "г", "б", "в", "з"]

# --- 2. Апостроф: три кодування ---------------------------------------------
# Правопис (§ 7, 138, 151) визначає вживання апострофа, але не кодову точку.
APOSTROPHES = [
    (chr(0x02BC), "primary", "Літера-модифікатор апостроф; рекомендоване кодування українського апострофа"),
    (chr(0x2019), "alternative", "Права одинарна лапка; типографська заміна апострофа"),
    (chr(0x0027), "alternative", "Машинописний апостроф ASCII; клавіатурна заміна"),
]

# --- 3. Розділові знаки (правопис 2019, розд. V, § 155–168) -------------------
# Перелік основних знаків — у вступі до розд. V; скісна риска — «крім них» (§ 165).
# (символ, нормативна назва, роль, параграф, примітка)
PUNCTUATION = [
    (".", "крапка", "primary", "§ 155", None),
    ("?", "знак питання", "primary", "§ 156", None),
    ("!", "знак оклику", "primary", "§ 157", None),
    (",", "кома", "primary", "§ 158", None),
    (";", "крапка з комою", "primary", "§ 159", None),
    (":", "двокрапка", "primary", "§ 160", None),
    (chr(0x2014), "тире", "primary", "§ 161", "Довге тире: так набрано заголовок § 161 «ТИРЕ (—)»"),
    (chr(0x2013), "тире", "alternative", "§ 161", "Коротке тире: поширена типографська заміна; так набрано § 161 у змісті видання"),
    (chr(0x2026), "три крапки", "primary", "§ 162", "Один знак U+2026; заголовок § 162 подає «(...)», тому варіант із трьох U+002E — у test_strings"),
    ("(", "дужки круглі", "primary", "§ 163", "Відкривна"),
    (")", "дужки круглі", "primary", "§ 163", "Закривна"),
    ("[", "дужки квадратні", "primary", "§ 163", "Відкривна"),
    ("]", "дужки квадратні", "primary", "§ 163", "Закривна"),
    ("<", "дужки кутові", "primary", "§ 163", "Відкривна; у правописі «кутові, або ламані (< >)», зокрема <…> для пропуску в цитаті"),
    (">", "дужки кутові", "primary", "§ 163", "Закривна"),
    (chr(0x00AB), "лапки", "primary", "§ 164", "Відкривні «лапки-ялинки» — рекомендовані зовнішні"),
    (chr(0x00BB), "лапки", "primary", "§ 164", "Закривні «лапки-ялинки»"),
    (chr(0x201E), "лапки", "primary", "§ 164", "Відкривні в парах „ “ і „ ”"),
    (chr(0x201C), "лапки", "primary", "§ 164", "Відкривні в парі “ ”, закривні в парі „ “ — «лапки-лапки», рекомендовані внутрішні"),
    (chr(0x201D), "лапки", "primary", "§ 164", "Закривні в парах “ ” і „ ”; форма „…” — традиційна в рукописних текстах (§ 164, п. 3)"),
    (chr(0x0022), "лапки", "alternative", "§ 164", "Прямі лапки ASCII, клавіатурна заміна"),
    ("-", "дефіс", "primary", "§ 37", "Як розділовий знак — у синтаксичних сполученнях (вступ до розд. V); U+002D — фактичне кодування"),
    (chr(0x2010), "дефіс", "alternative", "§ 37", "Типографський дефіс U+2010"),
    ("/", "скісна риска", "primary", "§ 165", "Розділовий знак (вступ до розд. V, § 165)"),
]

# --- 4. Історичні знаки XIX — поч. XX ст. -------------------------------------
# status: "core" — названі в пропозиції; "attested" — засвідчені в правописних
# системах XIX — поч. XX ст. Сторінки (с.) — за хрестоматією Німчука (2004, див.
# SOURCES_HISTORICAL); «Вікіпедія» — лише там, де в хрестоматії підтвердження не знайдено.
HISTORICAL = [
    ("ы", "core", "Єри; абетки Шевченка 1861 (с. 69) і Куліша 1857 (с. 75), Павловський 1818 (с. 40)"),
    ("ъ", "core", "Єр; абетки Шевченка 1861 (с. 69) і Куліша 1857 (с. 75), Павловський 1818 (с. 40)"),
    ("ѣ", "core", "Ять; абетки Шевченка і Куліша (с. 69, 75), Максимович (с. 89)"),
    ("ѳ", "core", "Фіта; абетки Шевченка 1861 (с. 69) і Куліша 1857 (с. 75)"),
    ("ѵ", "core", "Іжиця; абетки Шевченка і Куліша (с. 69, 75); Павловський: ще вживана на письмі (с. 40)"),
    ("ё", "attested", "Куліш, «Граматка» 1857: «сёго», «Ёго» (с. 75); слобожанський правопис (прим. упорядника, с. 68)"),
    ("э", "attested", "Абетки Шевченка 1861 (с. 69) і Куліша 1857 (с. 75)"),
    ("j", "attested", "Драгоманівка: латинська j для йотованих — ja, je, jy, ji (с. 161)"),
    ("ѯ", "attested", "Павловський 1818: «употребляютъ буквы Кси, Пси, Ижицу» (с. 40)"),
    ("ѱ", "attested", "Павловський 1818: «употребляютъ буквы Кси, Пси, Ижицу» (с. 40)"),
    ("ꙋ", "attested", "Гатцук, «Українська абетка» 1861: ук-монограф на місці у (с. 83); система не поширилася"),
    ("ѧ", "attested", "Гатцук 1861: юс малий на місці я (с. 83); система не поширилася"),
    # Максимовичівка: п'ять знаків для [і] — ѣ, ô, ê, ŷ, û (с. 89). Кириличних складених
    # символів немає, тому це послідовність «літера + U+0302».
    ("о" + chr(0x0302), "attested", "Максимович: [і] з етимологічного о, «нôсъ» (с. 89)"),
    ("е" + chr(0x0302), "attested", "Максимович: [і] з етимологічного е, «мêдъ» (с. 89)"),
    ("у" + chr(0x0302), "attested", "Максимович: [і] з етимологічного у, «замŷжъ» (с. 89)"),
    ("и" + chr(0x0302), "attested", "Максимович: [і] з етимологічного и, «добрû» (с. 89)"),
    # Знаки наголосу в текстах XIX ст.; носій — «а», бо важить сам комбінований знак.
    ("а" + chr(0x0301), "attested", "Наголос (акут, U+0301): Павловський 1818 (с. 40), Шевченко 1861 (с. 69), Куліш 1857 (с. 75)"),
    ("а" + chr(0x0300), "attested", "Наголос (гравіс, U+0300): Шевченко, «Буквар» 1861 (с. 69)"),
    ("g", "attested", "Кулішівка 1856–57: латинська g на позначення ґ (Вікіпедія; у переглянутих сторінках хрестоматії не знайдено)"),
]

# Кандидати попередніх версій, вилучені з інвентарю, з причиною.
EXCLUDED_HISTORICAL = {
    "ѕ ѡ ѫ ѹ": "Гатцук 1861 перелічує їх серед вилучених із церковної абетки (с. 83); у системах XIX ст. не вживаються",
    "ј (кирилична)": "Драгоманівка вживала латинську j (с. 161) — див. запис j",
    "а̂ і̂": "Максимович: серед знаків для [і] не названі (с. 89)",
    "надрядкові знаки Гатцука": "Система 1861 р. мала багато надрядкових знаків і не поширилася (с. 82); до інвентарю взято лише ꙋ і ѧ",
}

SOURCES_HISTORICAL = [
    "Німчук В. В., Пуряєва Н. В. (упор.) Історія українського правопису: XVI–XX століття. Хрестоматія. К.: Наукова думка, 2004. "
    "Електронна копія: Електронна бібліотека Інституту історії України НАН України, resource.history.org.ua/item/0007903",
    "Вікіпедія: Кулішівка (латинська g; переглянуто 23.10.2026)",
]

QUICK_TEST = "ґєії" + chr(0x02BC) + " ыъѣ"
PANGRAM = "Чуєш їх, доцю, га? Кумедна ж ти, прощайся без ґольфів!"


def codepoints(s):
    return [f"U+{ord(c):04X}" for c in s]


def names(s):
    return [unicodedata.name(c, "<unnamed>") for c in s]


def describe(s):
    """Поля, однакові для всіх записів: кодові точки й поведінка нормалізації."""
    nfd = unicodedata.normalize("NFD", s)
    nfc = unicodedata.normalize("NFC", s)
    return {
        "char": s,
        "codepoints": codepoints(s),
        "unicode_names": names(s),
        "n_codepoints": len(s),
        "nfc_equal": nfc == s,
        "nfd_codepoints": codepoints(nfd) if nfd != s else None,
    }


def letter_pair(lower, group, extra):
    rows = []
    upper = lower.upper()
    for ch, case, pair in ((lower, "lower", upper), (upper, "upper", lower)):
        row = describe(ch)
        row.update(group=group, case=case, case_pair=pair)
        row.update(extra(ch, case))
        rows.append(row)
    return rows


def build():
    entries = []

    for i, lo in enumerate(ALPHABET_LOWER, start=1):
        entries += letter_pair(
            lo, "alphabet",
            lambda ch, case, i=i, lo=lo: {
                "alphabet_index": i,
                "key_handwritten_form": case == "lower" and lo in KEY_HANDWRITTEN,
            },
        )

    for ch, role, note in APOSTROPHES:
        row = describe(ch)
        row.update(group="apostrophe", case=None, role=role, note=note)
        entries.append(row)

    for ch, name, role, section, note in PUNCTUATION:
        row = describe(ch)
        row.update(group="punctuation", case=None, mark_name=name, role=role,
                   pravopys_section=section, note=note)
        entries.append(row)

    for lo, status, note in HISTORICAL:
        entries += letter_pair(
            lo, "historical",
            lambda ch, case, status=status, note=note: {
                "status": status, "note": note,
            },
        )

    for n, row in enumerate(entries, start=1):
        row["id"] = f"C{n:03d}"

    return {
        "inventory_version": INVENTORY_VERSION,
        "built": date.today().isoformat(),
        "unicode_version": unicodedata.unidata_version,
        "sources": {
            "alphabet": "Сучасна українська абетка, 33 літери",
            "apostrophe_punctuation": "Український правопис (2019): § 7, 138, 151 (апостроф); розд. V, § 155–168 (розділові знаки)",
            "historical": SOURCES_HISTORICAL,
            "key_handwritten_form": "Чинні прописи для 1 класу (обраний еталон)",
        },
        "groups": {
            "alphabet": "33 літери в обох регістрах (66 знаків)",
            "apostrophe": "Апостроф у трьох кодуваннях",
            "punctuation": "Розділові знаки правопису 2019; role=alternative — поширені кодові заміни",
            "historical": "status=core — названі в пропозиції; attested — засвідчені в системах XIX — поч. XX ст. (сторінки хрестоматії Німчука в note)",
        },
        "excluded_historical": EXCLUDED_HISTORICAL,
        "test_strings": {
            "quick": QUICK_TEST,
            "pangram": PANGRAM,
            "apostrophe_variants": " ".join(f"п{a}ять м{a}ята" for a, _, _ in APOSTROPHES),
            "punctuation_line": (
                "Він сказав: «Іди сюди — „зараз“ (не завтра)!» Що?! Ну… Ну... "
                "Добре, так; гаразд-гаразд [див. <…>] і/або “інше” „рукописне”."
            ),
            "historical_line": "сынъ свѣтъ ѳеодоръ ѵпостась ёго эхо jак ѱалтирь ꙋкраїна ѧкъ мо" + chr(0x0302) + "й до" + chr(0x0301) + "бре",
            "all_chars": "".join(e["char"] for e in entries),
        },
        "entries": entries,
    }


def validate(inv):
    e = inv["entries"]
    by = lambda g: [x for x in e if x["group"] == g]
    alpha = by("alphabet")
    assert len(alpha) == 66, len(alpha)
    assert len({x["char"] for x in alpha}) == 66
    assert all(len(x["char"]) == 1 for x in alpha)
    assert sum(x["key_handwritten_form"] for x in alpha) == len(KEY_HANDWRITTEN) == 9
    assert len(by("apostrophe")) == 3
    assert inv["test_strings"]["quick"][4] == chr(0x02BC)
    chars = [x["char"] for x in e]
    assert len(chars) == len(set(chars)), "дублікати в інвентарі"
    missing = set(ALPHABET_LOWER) - set(inv["test_strings"]["pangram"].lower())
    assert not missing, f"панграма не покриває: {missing}"
    punct = {x["char"] for x in by("punctuation")}
    missing = punct - set(inv["test_strings"]["punctuation_line"]) - {chr(0x2013), chr(0x2010), chr(0x0022)}
    assert not missing, f"рядок розділових знаків не покриває: {missing}"
    for x in e:  # кожен символ має назву в базі Unicode
        assert "<unnamed>" not in x["unicode_names"], x


if __name__ == "__main__":
    inv = build()
    validate(inv)
    OUT.write_text(json.dumps(inv, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    counts = {}
    for x in inv["entries"]:
        key = x["group"] + (f"/{x['status']}" if x["group"] == "historical" else "")
        counts[key] = counts.get(key, 0) + 1
    print(f"{OUT.name}: {len(inv['entries'])} записів", counts, file=sys.stderr)

In [ ]:
%%writefile inventory/load.py
"""Спільний завантажувач інвентарю для скриптів аудиту."""

import json
from pathlib import Path

PATH = Path(__file__).with_name("charset_inventory.json")


def load_inventory(groups=None, path=PATH):
    """Повертає (метадані, записи).

    groups — набір груп (alphabet, apostrophe, punctuation, historical) або None для всіх.
    """
    inv = json.loads(Path(path).read_text(encoding="utf-8"))
    entries = [e for e in inv["entries"] if groups is None or e["group"] in groups]
    return inv, entries

In [ ]:
# Будуємо й перевіряємо інвентар (потрібен і для кроку 7)
!python3 inventory/build_inventory.py

## Крок 2. Фіксація версій

`versions/manifest.json` — що фіксуємо (правило: найновіший коміт на 23.10.2026, заморожений повним SHA). `versions/pinned.json` — знімок, зроблений 23.10.2026 (скорочена версія: ті самі SHA й ревізії, лише потрібні списки файлів). Скрипт `scripts/pin_versions.py` записано для відтворюваності, але **не запускаємо**: він перезаписав би знімок поточними ревізіями Hugging Face.

In [ ]:
%%writefile versions/manifest.json
{
  "about": "Що фіксуємо для аудиту статті 1. Правило: найновіший коміт на дату аудиту (23.10.2026), заморожений повним SHA; короткі коміти з пропозиції розв'язано через GitHub API. Скрипт scripts/pin_versions.py перевіряє їх і записує versions/pinned.json.",
  "git": [
    {
      "name": "GANwriting",
      "repo": "omni-us/research-GANwriting",
      "role": "method",
      "commit": "9e0d8a3a8327f00c67029dbf4a2fc1b0a88f730d",
      "commit_in_proposal": "9e0d8a3"
    },
    {
      "name": "HiGAN+",
      "repo": "ganji15/HiGANplus",
      "role": "method",
      "commit": "5d37de733f79e2a72a79c0f66e1865329ff56b4e",
      "commit_in_proposal": "5d37de7"
    },
    {
      "name": "HWT",
      "repo": "ankanbhunia/Handwriting-Transformers",
      "role": "method",
      "commit": "fbc66bf33da446fbb32ca19aaa403b6f1d28dbe7",
      "commit_in_proposal": "fbc66bf"
    },
    {
      "name": "VATr",
      "repo": "aimagelab/VATr",
      "role": "method",
      "commit": "183501e02113a3b4b636f2a2c5443771501c8f2b",
      "commit_in_proposal": "183501e"
    },
    {
      "name": "VATr++",
      "repo": "EDM-Research/VATr-pp",
      "role": "method",
      "commit": "a32d6fd39d74fecbc3c99dcbcfc397c17fc9fc32",
      "commit_in_proposal": "a32d6fd"
    },
    {
      "name": "WordStylist",
      "repo": "koninik/WordStylist",
      "role": "method",
      "commit": "03bb684e0eb26c42bcfc2ed719e73d60db888226",
      "commit_in_proposal": "03bb684"
    },
    {
      "name": "One-DM",
      "repo": "dailenson/One-DM",
      "role": "method",
      "commit": "dde2205a70a2c70d1786503d198a795358c80ee4",
      "commit_in_proposal": "dde2205"
    },
    {
      "name": "DiffusionPen",
      "repo": "koninik/DiffusionPen",
      "role": "method",
      "commit": "059e2f2bcb7372231038719e955597917c1a8783",
      "commit_in_proposal": "059e2f2"
    },
    {
      "name": "Eruku",
      "repo": "Blowing-Up-Groundhogs/Eruku",
      "role": "method",
      "note": "Найновіша версія на 23.10.2026: ліцензія MIT, виправлення style context (09.09.2026)",
      "commit": "3bcddbd1dc65ff9c8ccea96830982b61b2986d3d"
    },
    {
      "name": "Eruku (aimagelab, версія пропозиції)",
      "repo": "aimagelab/Eruku",
      "role": "related",
      "note": "Версія, переглянута в пропозиції (ced978a); без файлу ліцензії. Для порівняння підготовки тексту",
      "commit": "ced978aae0f4277eacb89dc95f9442922ee345c8",
      "commit_in_proposal": "ced978a"
    },
    {
      "name": "Emuru",
      "repo": "aimagelab/Emuru-autoregressive-text-img",
      "role": "method",
      "note": "У пропозиції коміт не вказано; зафіксовано HEAD на 23.10.2026",
      "commit": "d8db91a606297ae0b173ec90de03e31fd927ba91"
    },
    {
      "name": "google/fonts",
      "repo": "google/fonts",
      "role": "fonts",
      "commit": "e7db6ca603fd4862010383ff17ffd6cc4ffac1c0",
      "commit_in_proposal": "47d54ca1",
      "note": "Найновіший коміт на 23.10.2026; у пропозиції — 47d54ca1 (16.10.2026), підрахунок 355/15 перераховуємо на кроці 5"
    },
    {
      "name": "ukrainian-handwriting-synth",
      "repo": "ValeriiSielikhov/ukrainian-handwriting-synth",
      "role": "synthesis_tool",
      "commit": "ce2022f3fbd6b666f4efaa2e16858628a1866318",
      "note": "Інструмент синтетики для української: 35 рукописних шрифтів у fonts/; рендеринг рядка через Pillow (src/ukr_synth/renderer.py)"
    },
    {
      "name": "doc_gen",
      "repo": "mhlzcu/doc_gen",
      "role": "synthesis_tool",
      "commit": "40062f0189a618ee12a50cfb442669b841800165",
      "note": "Gruber et al. 2023, синтетика кириличного рукопису; 7 шрифтів у корені; рендеринг по одному символу (document_generator.py:361)"
    }
  ],
  "huggingface": [
    {
      "id": "konnik/DiffusionPen",
      "kind": "model",
      "used_by": [
        "DiffusionPen"
      ],
      "role": "weights"
    },
    {
      "id": "kdonitz/diffusionpen-ukrainian",
      "kind": "model",
      "used_by": [
        "Українська DiffusionPen"
      ],
      "role": "weights"
    },
    {
      "id": "kdonitz/diffusionpen-ukrainian",
      "kind": "space",
      "used_by": [
        "Українська DiffusionPen"
      ],
      "role": "inference code (app.py, generate_sentence.py)"
    },
    {
      "id": "google/canine-c",
      "kind": "model",
      "used_by": [
        "DiffusionPen",
        "Українська DiffusionPen"
      ],
      "role": "text encoder / tokenizer"
    },
    {
      "id": "stable-diffusion-v1-5/stable-diffusion-v1-5",
      "kind": "model",
      "used_by": [
        "DiffusionPen",
        "WordStylist",
        "One-DM"
      ],
      "role": "VAE, scheduler"
    },
    {
      "id": "runwayml/stable-diffusion-v1-5",
      "kind": "model",
      "used_by": [
        "One-DM"
      ],
      "role": "VAE (шлях у коді One-DM)"
    },
    {
      "id": "blowing-up-groundhogs/emuru",
      "kind": "model",
      "used_by": [
        "Emuru"
      ],
      "role": "weights"
    },
    {
      "id": "blowing-up-groundhogs/emuru_vae",
      "kind": "model",
      "used_by": [
        "Emuru",
        "Eruku"
      ],
      "role": "VAE"
    },
    {
      "id": "Blowing-Up-Groundhogs/Eruku",
      "kind": "model",
      "used_by": [
        "Eruku"
      ],
      "role": "weights"
    },
    {
      "id": "google/byt5-small",
      "kind": "model",
      "used_by": [
        "Emuru",
        "Eruku"
      ],
      "role": "tokenizer"
    },
    {
      "id": "blowing-up-groundhogs/vatrpp",
      "kind": "model",
      "used_by": [
        "VATr++"
      ],
      "role": "weights (посилання з README VATr)"
    }
  ],
  "weights_outside_hf": [
    {
      "method": "HiGAN+",
      "url": "https://github.com/ganji15/HiGANplus/tree/5d37de733f79e2a72a79c0f66e1865329ff56b4e/HiGAN%2B/pretrained",
      "note": "Ваги в самому репозиторії (HiGAN+/pretrained/deploy_HiGAN+.pth) — зафіксовано комітом; у релізах лише дані IAM"
    },
    {
      "method": "HWT",
      "url": "https://drive.google.com/file/d/16g9zgysQnWk7-353_tMig92KsZsrcM6k/view",
      "note": "Google Drive: ревізії немає, фіксуємо SHA-256 файлу"
    },
    {
      "method": "VATr",
      "url": "https://drive.google.com/drive/folders/1FGJe2uCuK8T9HrFzY_Zc-KMIo0oPJGGY",
      "note": "Google Drive: фіксуємо SHA-256 файлів"
    },
    {
      "method": "VATr++",
      "url": "https://drive.google.com/drive/folders/13rJhjl7VsyiXlPTBvnp1EKkKEhckLalr",
      "note": "Google Drive: фіксуємо SHA-256 файлів"
    },
    {
      "method": "WordStylist",
      "url": "https://drive.google.com/drive/folders/15jdDCoYuWAohKW_OEjD2LXWce0pxM7ux",
      "note": "Google Drive: фіксуємо SHA-256 файлів"
    },
    {
      "method": "One-DM",
      "url": "https://drive.google.com/drive/folders/108TB-z2ytAZSIEzND94dyufybjpqVyn6",
      "note": "Google Drive: фіксуємо SHA-256 файлів"
    },
    {
      "method": "GANwriting",
      "url": null,
      "note": "Ваги не опубліковано: README пропонує навчити модель самостійно"
    }
  ],
  "external": [
    {
      "name": "Kobzar KS",
      "version": "1.020",
      "url": "https://web.archive.org/web/20150123034457id_/http://kobzar.kyivstar.ua/main/assets/files/kobzarKS.zip",
      "sha256_zip": "b9c1ccb1c99a1812156914cd5722ab7515909f4acd3604c03a6bd687d584117e",
      "file": "KobzarKS_v1-020.otf",
      "sha256_file": "ff64b568b5ecd601be27b5012a1f0181bf386245e33358d8efef60144d7c8a7f",
      "license_url": "https://web.archive.org/web/20140922091814id_/http://kobzar.kyivstar.ua:80/main/assets/files/kobzarks_license-uk.pdf",
      "note": "Офіційний сайт kobzar.kyivstar.ua недоступний; архів — знімок Wayback Machine 23.01.2015. Ліцензія ПрАТ «Київстар»: безоплатна, невиключна; заборонено модифікацію і розбір (1.3.2, 3.1), вбудовування в ПЗ (1.3.1), використання ШІ у зв'язку з користуванням шрифтом (3.2). Файл шрифту в репозиторій аудиту не включаємо"
    }
  ]
}

In [ ]:
%%writefile versions/pinned.json
{"about":"Скорочений pinned.json для Colab (крок 7): ті самі коміти й ревізії, лише потрібні списки файлів","checked_at":"2026-10-23T21:09:37+00:00","git":[{"name":"GANwriting","repo":"omni-us/research-GANwriting","role":"method","commit":"9e0d8a3a8327f00c67029dbf4a2fc1b0a88f730d","status":"ok"},{"name":"HiGAN+","repo":"ganji15/HiGANplus","role":"method","commit":"5d37de733f79e2a72a79c0f66e1865329ff56b4e","status":"ok"},{"name":"HWT","repo":"ankanbhunia/Handwriting-Transformers","role":"method","commit":"fbc66bf33da446fbb32ca19aaa403b6f1d28dbe7","status":"ok"},{"name":"VATr","repo":"aimagelab/VATr","role":"method","commit":"183501e02113a3b4b636f2a2c5443771501c8f2b","status":"ok"},{"name":"VATr++","repo":"EDM-Research/VATr-pp","role":"method","commit":"a32d6fd39d74fecbc3c99dcbcfc397c17fc9fc32","status":"ok"},{"name":"WordStylist","repo":"koninik/WordStylist","role":"method","commit":"03bb684e0eb26c42bcfc2ed719e73d60db888226","status":"ok"},{"name":"One-DM","repo":"dailenson/One-DM","role":"method","commit":"dde2205a70a2c70d1786503d198a795358c80ee4","status":"ok"},{"name":"DiffusionPen","repo":"koninik/DiffusionPen","role":"method","commit":"059e2f2bcb7372231038719e955597917c1a8783","status":"ok"},{"name":"Eruku","repo":"Blowing-Up-Groundhogs/Eruku","role":"method","commit":"3bcddbd1dc65ff9c8ccea96830982b61b2986d3d","status":"ok"},{"name":"Eruku (aimagelab, версія пропозиції)","repo":"aimagelab/Eruku","role":"related","commit":"ced978aae0f4277eacb89dc95f9442922ee345c8","status":"ok"},{"name":"Emuru","repo":"aimagelab/Emuru-autoregressive-text-img","role":"method","commit":"d8db91a606297ae0b173ec90de03e31fd927ba91","status":"ok"},{"name":"google/fonts","repo":"google/fonts","role":"fonts","commit":"e7db6ca603fd4862010383ff17ffd6cc4ffac1c0","status":"ok"},{"name":"ukrainian-handwriting-synth","repo":"ValeriiSielikhov/ukrainian-handwriting-synth","role":"synthesis_tool","commit":"ce2022f3fbd6b666f4efaa2e16858628a1866318","status":"ok"},{"name":"doc_gen","repo":"mhlzcu/doc_gen","role":"synthesis_tool","commit":"40062f0189a618ee12a50cfb442669b841800165","status":"ok"}],"huggingface":[{"id":"konnik/DiffusionPen","kind":"model","sha":"5636a68365001435ec76553114313c22c6f1ca07","status":"ok","resolved_id":"konnik/DiffusionPen","license":"mit"},{"id":"kdonitz/diffusionpen-ukrainian","kind":"model","sha":"a96778d870cf357b71276f4f275b2f4dfaa6011d","status":"ok","resolved_id":"kdonitz/diffusionpen-ukrainian","license":null},{"id":"kdonitz/diffusionpen-ukrainian","kind":"space","sha":"8815298140b09307ef215d83fb2c803d984c9b20","status":"ok","resolved_id":"kdonitz/diffusionpen-ukrainian","license":"apache-2.0","style_ref_dirs":["0000","0002","0004","0005","0006","0007","0011","0013","0014","0017","0018","0020","0021","0023","0025","0027","0028","0032","0033","0034","0035","0037","0038","0041","0047","0048","0050","0056","0058","0060","0061","0063","0064","0065","0066","0067","0069","0071","0073","0075","0076","0077","0078","0079","0080","0081","0082","0086","0087","0089","0090","0091","0092","0093","0094","0095","0096","0099","0100","0101","0102","0104","0105","0108","0110","0111","0112","0114","0115","0118","0120","0123","0127","0129","0132","0133","0134","0135","0136","0137","0138","0142","0144","0150","0152","0153","0154","0156","0157","0158","0159","0160","0161","0162","0163","0164","0165","0167","0169","0171","0172","0173","0174","0175","0176","0178","0179","0181","0182","0184","0185","0186","0187","0188","0189","0191","0196","0197","0198","0199","0200","0201","0202","0213","0214","0215","0218","0219","0220","0221","0223","0224","0225","0228","0231","0233","0235","0402","0403","0404","0405","0406","0407","0408","0409","0410","0411","0412","0413","0414","0416","0417","0418","0419","0420","0421","0422","0423","0424","0425","0427","0429","0430","0434","0436","0437","0438","0439","0440","0442","0447","0448","0449","0450","0451","0452","0453","0454","0455","0456","0458","0459","0460","0461","0462","0463","0464","0465","0466","0467","0468","0470","0472","0473","0474","0475","0476","0477","0478","0479","0482","0483","0484","0485","0486","0488","0489","0490","0491","0492","0494","0495","0496","0497","0498","0499","0500","0503","0504","0506","0508","0509","0511","0512","0513","0514","0515","0540","0541","0543","0545","0546","0547","0548","0549","0551","0555","0556","0558","0559","0560","0563","0565","0566","0568","0569","0570","0579","0581","0582","0586","0587","0588","0589","0590","0592","0593","0594","0595","0596","0597","0598","0599","0600","0623","0624","0628","0629","0630","0631","0633","0636","0637","0640","0641","0642","0644","0647","0648","0649","0650","0651","0652","0653","0654","0655","0656","0657","0660","0661","0662","0665","0666","0667","0671","0672","0676","0680","0681","0683","0688","0693","0694","0697","0700","0702","0703","5052"],"files":[{"path":"README.md","size":1044,"lfs_sha256":null},{"path":"app.py","size":18647,"lfs_sha256":null},{"path":"extract_style_refs.py","size":1373,"lfs_sha256":null},{"path":"feature_extractor.py","size":610,"lfs_sha256":null},{"path":"generate_sentence.py","size":73709,"lfs_sha256":null},{"path":"requirements.txt","size":181,"lfs_sha256":null},{"path":"style_refs/0000/a01-219-0000-05-w03.png","size":10634,"lfs_sha256":null},{"path":"style_refs/0000/a01-219-0000-09-w01.png","size":11678,"lfs_sha256":null},{"path":"style_refs/0000/a01-219-0000-10-w00.png","size":7145,"lfs_sha256":null},{"path":"style_refs/0000/a01-379-0000-02-w02.png","size":3018,"lfs_sha256":null},{"path":"style_refs/0000/a01-379-0000-04-w03.png","size":18762,"lfs_sha256":null},{"path":"style_refs/0002/a01-019-0002-03-w02.png","size":16967,"lfs_sha256":null},{"path":"style_refs/0002/a01-024-0002-03-w02.png","size":3478,"lfs_sha256":null},{"path":"style_refs/0002/a01-029-0002-07-w01.png","size":5214,"lfs_sha256":null},{"path":"style_refs/0002/a01-033-0002-04-w03.png","size":8672,"lfs_sha256":null},{"path":"style_refs/0002/a01-035-0002-01-w02.png","size":3116,"lfs_sha256":null},{"path":"unet.py","size":49099,"lfs_sha256":null},{"path":"utils/__init__.py","size":0,"lfs_sha256":null},{"path":"utils/auxilary_functions.py","size":3679,"lfs_sha256":null},{"path":"utils/word_cleanup_nafnet.py","size":2710,"lfs_sha256":null},{"path":"utils/word_dataset.py","size":18412,"lfs_sha256":null},{"path":"punct_bank/colon/colon_0002_00035.png","size":522,"lfs_sha256":null},{"path":"punct_bank/colon/colon_0004_00054.png","size":316,"lfs_sha256":null},{"path":"punct_bank/comma/comma_0002_00353.png","size":336,"lfs_sha256":null},{"path":"punct_bank/comma/comma_0002_00354.png","size":318,"lfs_sha256":null},{"path":"punct_bank/dash/dash_0006_00304.png","size":358,"lfs_sha256":null},{"path":"punct_bank/dash/hyphen_0005_00200.png","size":287,"lfs_sha256":null},{"path":"punct_bank/exclaim/exclaim_0034_00000.png","size":549,"lfs_sha256":null},{"path":"punct_bank/exclaim/exclaim_0078_00002.png","size":639,"lfs_sha256":null},{"path":"punct_bank/hyphen/hyphen_0002_00096.png","size":231,"lfs_sha256":null},{"path":"punct_bank/hyphen/hyphen_0005_00187.png","size":323,"lfs_sha256":null},{"path":"punct_bank/period/period_0002_00137.png","size":102,"lfs_sha256":null},{"path":"punct_bank/period/period_0002_00148.png","size":122,"lfs_sha256":null},{"path":"punct_bank/question/question_0020_00012.png","size":654,"lfs_sha256":null},{"path":"punct_bank/question/question_0020_00024.png","size":603,"lfs_sha256":null},{"path":"punct_bank/semicolon/semicolon_0005_00023.png","size":525,"lfs_sha256":null},{"path":"punct_bank/semicolon/semicolon_0005_00024.png","size":520,"lfs_sha256":null}]},{"id":"google/canine-c","kind":"model","sha":"dc0eaffdff3fa9161613311c7096eeb3e133ee19","status":"ok","resolved_id":"google/canine-c","license":"apache-2.0"},{"id":"stable-diffusion-v1-5/stable-diffusion-v1-5","kind":"model","sha":"451f4fe16113bff5a5d2269ed5ad43b0592e9a14","status":"ok","resolved_id":"stable-diffusion-v1-5/stable-diffusion-v1-5","license":"creativeml-openrail-m"},{"id":"runwayml/stable-diffusion-v1-5","kind":"model","sha":"451f4fe16113bff5a5d2269ed5ad43b0592e9a14","status":"ok","resolved_id":"stable-diffusion-v1-5/stable-diffusion-v1-5","license":"creativeml-openrail-m"},{"id":"blowing-up-groundhogs/emuru","kind":"model","sha":"c75b78dd6ea0306123ad5a12b18b9ae4a927e149","status":"ok","resolved_id":"blowing-up-groundhogs/emuru","license":"mit"},{"id":"blowing-up-groundhogs/emuru_vae","kind":"model","sha":"139f13a8ba9de13c318266c34bcaf71f9b877df4","status":"ok","resolved_id":"blowing-up-groundhogs/emuru_vae","license":"mit"},{"id":"Blowing-Up-Groundhogs/Eruku","kind":"model","sha":"13f4f7501f0f1d7270da08fded134c57d3635705","status":"ok","resolved_id":"blowing-up-groundhogs/eruku","license":"apache-2.0"},{"id":"google/byt5-small","kind":"model","sha":"68377bdc18a2ffec8a0533fef03b1c513a4dd49d","status":"ok","resolved_id":"google/byt5-small","license":"apache-2.0"},{"id":"blowing-up-groundhogs/vatrpp","kind":"model","sha":"fb73be2be8e96ce70d9cad9675a30667e19ac376","status":"ok","resolved_id":"blowing-up-groundhogs/vatrpp","license":"mit","files":[{"path":"files/unifont.pickle","size":16978300,"lfs_sha256":"0804979068f0d169b343fbe0fe8d7ff478165d07a671fcf52e20f625db8e7f9f"}]}],"external":[{"name":"Kobzar KS","version":"1.020","url":"https://web.archive.org/web/20150123034457id_/http://kobzar.kyivstar.ua/main/assets/files/kobzarKS.zip","sha256_zip":"b9c1ccb1c99a1812156914cd5722ab7515909f4acd3604c03a6bd687d584117e","file":"KobzarKS_v1-020.otf","sha256_file":"ff64b568b5ecd601be27b5012a1f0181bf386245e33358d8efef60144d7c8a7f","license_url":"https://web.archive.org/web/20140922091814id_/http://kobzar.kyivstar.ua:80/main/assets/files/kobzarks_license-uk.pdf","note":"Офіційний сайт kobzar.kyivstar.ua недоступний; архів — знімок Wayback Machine 23.01.2015. Ліцензія ПрАТ «Київстар»: безоплатна, невиключна; заборонено модифікацію і розбір (1.3.2, 3.1), вбудовування в ПЗ (1.3.1), використання ШІ у зв'язку з користуванням шрифтом (3.2). Файл шрифту в репозиторій аудиту не включаємо"}]}

In [ ]:
%%writefile scripts/pin_versions.py
"""Фіксує версії коду й моделей для аудиту (стаття 1, крок 2).

Читає versions/manifest.json і записує versions/pinned.json:
- git: перевіряє, що кожен коміт є на GitHub (git завантажує лише об'єкт коміту,
  без дерева файлів), бере його дату й заголовок, порівнює з поточним HEAD;
- Hugging Face: поточна ревізія (sha) моделі чи Space, ліцензія, список файлів з SHA-256.
Потрібні лише git і стандартна бібліотека Python: працює і локально, і в Colab.
Ліцензії репозиторіїв беруться з GitHub API (без токена — 60 запитів/год; GITHUB_TOKEN знімає ліміт).

Запуск: python3 scripts/pin_versions.py
"""

import json
import os
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
MANIFEST = ROOT / "versions" / "manifest.json"
OUT = ROOT / "versions" / "pinned.json"


def get_json(url, headers=None):
    req = urllib.request.Request(url, headers={"User-Agent": "ukr-htg-audit", **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.load(r), None
    except urllib.error.HTTPError as e:
        return None, f"HTTP {e.code}"
    except urllib.error.URLError as e:
        return None, str(e.reason)


def github_license(repo):
    headers = {"Accept": "application/vnd.github+json"}
    if os.environ.get("GITHUB_TOKEN"):
        headers["Authorization"] = f"Bearer {os.environ['GITHUB_TOKEN']}"
    info, err = get_json(f"https://api.github.com/repos/{repo}", headers)
    if err:
        return None, err
    return (info.get("license") or {}).get("spdx_id"), None


def git(*args, cwd=None):
    out = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, timeout=300)
    return out.stdout.strip() if out.returncode == 0 else None


def pin_git(item):
    repo, sha = item["repo"], item["commit"]
    url = f"https://github.com/{repo}.git"
    row = dict(item)
    with tempfile.TemporaryDirectory() as tmp:
        git("init", "-q", "--bare", tmp)
        fetched = git("fetch", "-q", "--depth", "1", "--filter=tree:0", url, sha, cwd=tmp)
        if fetched is None:
            return row | {"status": "error", "error": f"коміт {sha} не завантажується з {url}"}
        fmt = git("log", "-1", "--format=%H%n%cI%n%P%n%s", sha, cwd=tmp).split("\n", 3)
    full, date, parents, subject = (fmt + [""] * 4)[:4]
    if full != sha:
        return row | {"status": "error", "error": f"очікували {sha}, отримали {full}"}
    row["pinned"] = {"sha": full, "committer_date": date,
                     "is_merge": len(parents.split()) > 1, "subject": subject}
    head = (git("ls-remote", url, "HEAD") or "").split()
    row["head_at_check"] = head[0] if head else None
    row["head_moved"] = bool(head) and head[0] != sha
    row["license"], err = github_license(repo)
    if err:
        row["license_error"] = err
    row["status"] = "ok"
    return row


def pin_hf(item):
    kind, rid = item["kind"], item["id"]
    row = dict(item)
    data, err = get_json(f"https://huggingface.co/api/{kind}s/{rid}?blobs=true")
    if err:
        return row | {"status": "unavailable", "error": err}
    card = data.get("cardData") or {}
    row.update(
        status="ok",
        resolved_id=data.get("id"),
        sha=data.get("sha"),
        last_modified=data.get("lastModified"),
        license=card.get("license"),
        gated=data.get("gated"),
        disabled=data.get("disabled"),
        runtime_stage=(data.get("runtime") or {}).get("stage"),
        files=[
            {"path": s["rfilename"], "size": s.get("size"),
             "lfs_sha256": (s.get("lfs") or {}).get("sha256")}
            for s in data.get("siblings", [])
        ],
    )
    return row


def main():
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    out = {
        "checked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "git": [pin_git(x) for x in manifest["git"]],
        "huggingface": [pin_hf(x) for x in manifest["huggingface"]],
        "weights_outside_hf": manifest["weights_outside_hf"],
        "external": manifest.get("external", []),
    }
    OUT.write_text(json.dumps(out, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    for g in out["git"]:
        if g["status"] != "ok":
            print(f"GIT  {g['name']:<30} ПОМИЛКА {g['error']}")
            continue
        p = g["pinned"]
        notes = [n for n, on in (("злиття", p["is_merge"]), ("HEAD пішов далі", g["head_moved"])) if on]
        print(f"GIT  {g['name']:<30} {p['sha'][:12]} {p['committer_date'][:10]} "
              f"{g['license'] or g.get('license_error')} {', '.join(notes)}")
    for h in out["huggingface"]:
        if h["status"] != "ok":
            print(f"HF   {h['kind']:<6} {h['id']:<45} НЕДОСТУПНО {h['error']}")
            continue
        size = sum(f["size"] or 0 for f in h["files"]) / 2**20
        flags = " ".join(f for f, on in (("gated", h["gated"]), ("disabled", h["disabled"])) if on)
        stage = f" runtime={h['runtime_stage']}" if h["runtime_stage"] else ""
        moved = f" -> {h['resolved_id']}" if h["resolved_id"] != h["id"] else ""
        print(f"HF   {h['kind']:<6} {h['id']:<45} {h['sha'][:12]} {h['last_modified'][:10]} "
              f"{h['license']} {size:.0f} МБ {flags}{stage}{moved}")
    errors = [g for g in out["git"] if g["status"] != "ok"]
    print(f"\n{OUT.relative_to(ROOT)}: git {len(out['git']) - len(errors)}/{len(out['git'])}, "
          f"HF {sum(h['status'] == 'ok' for h in out['huggingface'])}/{len(out['huggingface'])}")
    return 1 if errors else 0


if __name__ == "__main__":
    sys.exit(main())

## Крок 3. Рівні 1–2 для методів (CPU)

Для кожного методу зонд виконує **справжній код репозиторію** на зафіксованому коміті: рівень 1 — кодувальник алфавіту чи токенізатор кодує й декодує кожен символ інвентарю; рівень 2 — тестові рядки проходять шлях підготовки тексту (для VATr — і `mytext.txt`, і `generator.py`). Де символ губиться, записано файл і рядок.

Статуси: `ok` — без змін; `lossy` — змінився; `lost` — мовчки вилучений; `error` — виняток у коді методу; `probe_error` — збій зонда.

`fetch_sources.py` завантажує код методів (потрібен і для кроку 7). Запуск зондів (`run_level12.py`) необов'язковий — результати вже є локально.

In [ ]:
%%writefile scripts/level12/common.py
"""Спільне для зондів рівнів 1–2 (стаття 1, крок 3).

Зонд виконує справжній код репозиторію методу: кодувальник алфавіту чи токенізатор
(рівень 1) і функції підготовки тексту (рівень 2). Нейромережі не потрібні: там, де шлях
тексту доходить до моделі, зонд підставляє заглушку, що лише записує отримані дані.

Статуси символу:
  ok     — символ пройшов без змін;
  lossy  — пройшов, але змінився (інший символ, інший регістр, розклад тощо);
  lost   — мовчки вилучений (фільтр, заміна на порожнє місце);
  error  — виняток (KeyError тощо); where — файл і рядок у репозиторії.
"""

import contextlib
import io
import json
import os
import pickle
import sys
import traceback
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "inventory"))
from load import load_inventory  # noqa: E402

# Код методів, ваги й шрифти (усе відтворюване) — поза текою проєкту
WORK = Path(os.environ.get("AUDIT_WORK", Path.home() / ".cache" / "ukr-htg-audit" / "work"))
REPOS = WORK / "repos"
RESULTS = ROOT / "results" / "level12"


class Captured(Exception):
    """Заглушка моделі кидає це, щоб зупинити шлях і повернути те, що до неї дійшло."""

    def __init__(self, payload):
        super().__init__("captured")
        self.payload = payload


class Site:
    """Місце в коді, де символ губиться без винятку. Перевіряємо, що рядок справді такий."""

    def __init__(self, repo_dir, file, line, contains):
        self.file, self.line = file, line
        text = (Path(repo_dir) / file).read_text(encoding="utf-8").splitlines()[line - 1]
        if contains not in text:
            raise AssertionError(f"{file}:{line} не містить {contains!r}: {text.strip()!r}")
        self.code = text.strip()

    @classmethod
    def find(cls, repo_dir, file, contains):
        """Те саме, але рядок шукаємо: у різних версіях репозиторію він зсувається."""
        lines = (Path(repo_dir) / file).read_text(encoding="utf-8").splitlines()
        hits = [i for i, text in enumerate(lines, start=1) if contains in text]
        if len(hits) != 1:
            raise AssertionError(f"{file}: {contains!r} знайдено {len(hits)} разів")
        return cls(repo_dir, file, hits[0], contains)

    def ref(self):
        return f"{self.file}:{self.line}"


def where_in_repo(exc, repo_dir):
    """Останній кадр трасування, що лежить у репозиторії методу."""
    repo_dir = str(Path(repo_dir).resolve())
    frames = [f for f in traceback.extract_tb(exc.__traceback__)
              if str(Path(f.filename).resolve()).startswith(repo_dir)]
    if not frames:
        return None
    f = frames[-1]
    return {"ref": f"{Path(f.filename).resolve().relative_to(repo_dir)}:{f.lineno}", "code": f.line}


@contextlib.contextmanager
def in_repo(repo_dir):
    """Репозиторії відкривають файли відносними шляхами, тож працюємо з їхнього кореня.

    Різні репозиторії мають однакові назви пакетів (util, models, train), тому після виходу
    вивантажуємо все, що імпортовано з цього репозиторію.
    """
    old_cwd, old_path = os.getcwd(), list(sys.path)
    root = str(Path(repo_dir).resolve())
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    try:
        yield
    finally:
        os.chdir(old_cwd)
        sys.path[:] = old_path
        for name, mod in list(sys.modules.items()):
            f = getattr(mod, "__file__", None) or ""
            if f and str(Path(f).resolve()).startswith(root):
                del sys.modules[name]


@contextlib.contextmanager
def quiet():
    with contextlib.redirect_stdout(io.StringIO()):
        yield


def safe_unpickle(path):
    """Читає pickle гліфів, дозволяючи лише класи numpy (перевірено pickletools)."""
    import numpy as np

    allowed = {("numpy", "ndarray"), ("numpy", "dtype"),
               ("numpy.core.multiarray", "_reconstruct"), ("numpy._core.multiarray", "_reconstruct")}

    class Restricted(pickle.Unpickler):
        def find_class(self, module, name):
            if (module, name) not in allowed:
                raise pickle.UnpicklingError(f"заборонено {module}.{name}")
            if name == "_reconstruct":
                return np._core.multiarray._reconstruct if hasattr(np, "_core") else np.core.multiarray._reconstruct
            return getattr(np, name)

    with open(path, "rb") as f:
        return Restricted(f).load()


def _error_row(exc, repo_dir):
    """Виняток у коді репозиторію — результат методу; поза ним — збій самого зонда."""
    where = where_in_repo(exc, repo_dir)
    return {"status": "error" if where else "probe_error",
            "error": f"{type(exc).__name__}: {exc}"[:200], "where": where}


def run_chars(entries, fn, repo_dir):
    """fn(char) -> (status, output, site|None). Винятки ловимо й прив'язуємо до рядка коду."""
    out = {}
    for e in entries:
        ch = e["char"]
        try:
            status, value, site = fn(ch)
            row = {"status": status, "output": value}
            if site is not None:
                row["where"] = {"ref": site.ref(), "code": site.code}
        except Exception as exc:  # noqa: BLE001 — саме виняток і є результатом
            row = _error_row(exc, repo_dir)
        out[e["id"]] = row
    return out


def compare(src, dst):
    """Що з src не дійшло до dst (посимвольно, з урахуванням порядку)."""
    import difflib
    sm = difflib.SequenceMatcher(a=src, b=dst, autojunk=False)
    lost, changed = [], []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == "delete":
            lost.append(src[i1:i2])
        elif op == "replace":
            changed.append([src[i1:i2], dst[j1:j2]])
    return lost, changed


def run_strings(strings, fn, repo_dir):
    """fn(text) -> (output_text, site|None) — проводимо тестові рядки через підготовку тексту."""
    out = {}
    for key, text in strings.items():
        try:
            result, site = fn(text)
            lost, changed = compare(text, result)
            row = {"input": text, "output": result, "lost": lost, "changed": changed,
                   "status": "ok" if not lost and not changed else "lost" if not changed else "lossy"}
            if site is not None and (lost or changed):
                row["where"] = {"ref": site.ref(), "code": site.code}
        except Exception as exc:  # noqa: BLE001
            row = {"input": text, **_error_row(exc, repo_dir)}
        out[key] = row
    return out


def test_strings(inv):
    return {k: v for k, v in inv["test_strings"].items() if k != "all_chars"}


def save(method, payload):
    RESULTS.mkdir(parents=True, exist_ok=True)
    path = RESULTS / f"{method}.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    return path


def pinned(name):
    data = json.loads((ROOT / "versions" / "pinned.json").read_text(encoding="utf-8"))
    for key in ("git", "huggingface"):
        for row in data[key]:
            if row.get("name") == name or row.get("id") == name:
                return row
    raise KeyError(name)


def inventory():
    return load_inventory()

In [ ]:
%%writefile scripts/level12/probes_alphabet.py
"""Зонди методів із закритим алфавітом: GANwriting, HiGAN+, HWT, VATr, VATr++, One-DM, WordStylist.

Кожен зонд імпортує кодувальник із репозиторію й викликає його так, як це робить шлях
генерації методу. Посилання на рядки коду перевіряє Site: якщо код зміниться, зонд упаде.
"""

import shutil
import types

from common import Captured, REPOS, Site, in_repo, quiet, run_chars, run_strings, safe_unpickle


def _roundtrip(encode, decode):
    def fn(ch):
        ids = encode(ch)
        back = decode(ids)
        return ("ok" if back == ch else "lossy"), back, None
    return fn


def probe_ganwriting(entries, strings):
    repo = REPOS / "GANwriting"
    with in_repo(repo), quiet():
        import load_data as ld
        ds = ld.IAM_words({}, oov=False)

        def encode(text):  # load_data.py:171 — той самий label_padding, що й у тестовому скрипті
            return ds.label_padding(text, ld.num_tokens)

        def decode(ids):
            body = ids[ids.index(ld.tokens["GO_TOKEN"]) + 1: ids.index(ld.tokens["END_TOKEN"])]
            return "".join(ld.index2letter[i - ld.num_tokens] for i in body)

        level1 = run_chars(entries, _roundtrip(encode, decode), repo)
        # Тестовий скрипт tt.test_single_writer.4_scenarios.py ділить корпус за пробілами
        # (рядок 31: _f.read().split()) і кодує кожне слово label_padding.
        level2 = run_strings(strings, lambda t: (" ".join(decode(encode(w)) for w in t.split()), None), repo)
    return {
        "encoder": "load_data.labelDictionary: 52 латинські літери (string.ascii_lowercase + ascii_uppercase)",
        "paths": {"tt.test_single_writer.4_scenarios.py → label_padding": {"level1": level1, "level2": level2}},
        "notes": ["Алфавіт без цифр і розділових знаків", f"MAX_CHARS = {ld.MAX_CHARS}: довші слова виходять за OUTPUT_MAX_LEN"],
    }


def probe_higan(entries, strings):
    repo = REPOS / "HiGANplus"
    src = repo / "HiGAN+"
    with in_repo(src), quiet():
        from lib.alphabet import Alphabets, strLabelConverter
        conv = strLabelConverter("all")

        def encode(text):  # networks/model.py:469/473 — encode(texts), texts = text.split(' ')
            labels, _ = conv.encode([text, text])  # список із >1 слова: шлях із pad_sequence
            return labels[0].tolist()[: len(text)]

        decode = lambda ids: "".join(conv.alphabet[i] for i in ids)
        level1 = run_chars(entries, _roundtrip(encode, decode), src)
        level2 = run_strings(strings, lambda t: (" ".join(decode(encode(w)) for w in t.split(" ") if w), None), src)
    return {
        "encoder": f"lib/alphabet.py Alphabets['all']: {len(Alphabets['all'])} символів (IAM)",
        "paths": {"networks/model.py eval_style/eval_rand → strLabelConverter.encode": {"level1": level1, "level2": level2}},
        "notes": ["word_capitalize (lib/alphabet.py:155) переводить першу літеру в ASCII через NFKD — лише для випадкових слів лексикону"],
    }


def probe_hwt(entries, strings):
    repo = REPOS / "HWT"
    with in_repo(repo), quiet():
        from params import ALPHABET
        from util.models.OCR_network import strLabelConverter
        conv = strLabelConverter(ALPHABET)

        def encode(text):  # util/models/model.py:520 — encode(self.text), слова в bytes
            ids, _ = conv.encode([text.encode()])
            return ids[0].tolist()

        decode = lambda ids: "".join(conv.alphabet[i - 1] for i in ids)
        level1 = run_chars(entries, _roundtrip(encode, decode), repo)
        site = Site(repo, "util/models/model.py", 520, "self.text = [j.encode() for j in sum([i.split(' ')")
        level2 = run_strings(strings, lambda t: (" ".join(decode(encode(w)) for w in t.split(" ") if w), site), repo)
    return {
        "encoder": f"params.py ALPHABET: {len(ALPHABET)} символів (IAM)",
        "paths": {"mytext.txt → util/models/model.py:520 → strLabelConverter.encode": {"level1": level1, "level2": level2}},
    }


def _vatr_glyphs(pickle_path, entries):
    symbols = {s["idx"][0] for s in safe_unpickle(pickle_path)}
    out = {}
    for e in entries:
        missing = [f"U+{ord(c):04X}" for c in e["char"] if ord(c) not in symbols]
        out[e["id"]] = {"status": "ok" if not missing else "missing", "missing": missing}
    return out, len(symbols)


def probe_vatr(entries, strings):
    repo = REPOS / "VATr"
    with in_repo(repo), quiet():
        from generator import FakeArgs
        from models.OCR_network import strLabelConverter
        args = FakeArgs()
        conv = strLabelConverter(args.alphabet + args.special_alphabet)  # models/model.py:363

        def gen_encode(text):  # generator.py:103 — netconverter.encode(text.split())
            ids, _, _ = conv.encode([text])
            return ids[0].tolist()

        decode = lambda ids: "".join(conv.alphabet[i - 1] for i in ids)
        gen1 = run_chars(entries, _roundtrip(gen_encode, decode), repo)
        gen2 = run_strings(strings, lambda t: (" ".join(decode(gen_encode(w)) for w in t.split()), None), repo)

        # Шлях mytext.txt: фільтр у models/model.py:410 виконуємо як є — збираємо модель
        # (без ваг) з тестовим mytext.txt і читаємо self.text.
        from models.model import VATr
        from models.unifont_module import UnifontModule
        # models/model.py:128 створює UnifontModule без device, а типове значення — 'cuda'.
        # Щоб зібрати модель на CPU, міняємо лише це типове значення.
        UnifontModule.__init__.__defaults__ = ("cpu",) + UnifontModule.__init__.__defaults__[1:]
        site = Site(repo, "models/model.py", 410, "''.join(c for c in self.text if c in (self.args.alphabet + self.args.special_alphabet))")
        args.device = "cpu"
        mytext = repo / "mytext.txt"
        original = mytext.read_bytes()

        def mytext_path(text):
            mytext.write_text(text, encoding="utf-8")
            try:
                model = VATr(args)
            finally:
                mytext.write_bytes(original)
            return " ".join(w.decode() for w in model.text), site

        def mytext_char(ch):
            out, s = mytext_path(f"a{ch}a")
            got = out[1:-1] if out.startswith("a") and out.endswith("a") else out
            return ("ok" if got == ch else "lost" if got == "" else "lossy"), got, (s if got != ch else None)

        my1 = run_chars(entries, mytext_char, repo)
        my2 = run_strings(strings, mytext_path, repo)
        glyphs, n = _vatr_glyphs(repo / "files" / "unifont.pickle", entries)
    return {
        "encoder": f"generator.py FakeArgs: alphabet {len(args.alphabet)} (IAM) + special_alphabet {len(args.special_alphabet)} (грецька)",
        "paths": {
            "generator.py → netconverter.encode": {"level1": gen1, "level2": gen2},
            "mytext.txt → models/model.py:410 → netconverter.encode": {"level1": my1, "level2": my2},
        },
        "glyphs": {"file": "files/unifont.pickle", "n_symbols": n, "entries": glyphs},
    }


def probe_vatrpp(entries, strings):
    repo = REPOS / "VATrplusplus"
    with in_repo(repo), quiet():
        from util.misc import FakeArgs
        from models.OCR_network import strLabelConverter
        from generate.writer import Writer
        args = FakeArgs()
        conv = strLabelConverter(args.alphabet + args.special_alphabet)  # models/model.py:219
        site = Site(repo, "generate/writer.py", 70, 'text = "".join([c for c in text if c in self.model.args.alphabet])')

        def model_stub(*a, **k):
            raise Captured(a)

        fake = types.SimpleNamespace(model=types.SimpleNamespace(
            args=args, netconverter=conv, _generate_fakes=model_stub))

        def writer_path(text):  # generate/writer.py:70–78, виконується справжня create_fake_sentence
            try:
                Writer.create_fake_sentence(fake, None, text)
            except Captured as c:
                text_encode, len_text = c.payload[1], c.payload[2]
                flat = [i for i in text_encode[0].flatten().tolist() if i]  # без нулів доповнення
                return " ".join(_split_words(flat, len_text.tolist(), conv)), site
            raise AssertionError("модель не викликано")

        def char(ch):
            out, s = writer_path(f"a{ch}a")
            got = out[1:-1] if out.startswith("a") and out.endswith("a") else out
            return ("ok" if got == ch else "lost" if got == "" else "lossy"), got, (s if got != ch else None)

        lvl1 = run_chars(entries, char, repo)
        lvl2 = run_strings(strings, writer_path, repo)
    glyphs, n = _vatr_glyphs(REPOS / "vatrpp-hf" / "files" / "unifont.pickle", entries)
    return {
        "encoder": f"util/misc.py FakeArgs: alphabet {len(args.alphabet)} (IAM) + special_alphabet {len(args.special_alphabet)}",
        "paths": {"generate.py → generate/writer.py:70 → netconverter.encode": {"level1": lvl1, "level2": lvl2}},
        "glyphs": {"file": "blowing-up-groundhogs/vatrpp: files/unifont.pickle", "n_symbols": n, "entries": glyphs},
    }


def _split_words(flat_ids, lengths, conv):
    words, pos = [], 0
    for n in lengths:
        words.append("".join(conv.alphabet[i - 1] for i in flat_ids[pos:pos + n]))
        pos += n
    return words


def probe_onedm(entries, strings):
    repo = REPOS / "One-DM"
    glyph_file = REPOS / "onedm-data" / "unifont.pickle"  # з English_data.zip, див. fetch_sources.py
    target = repo / "data" / "unifont.pickle"
    if not target.exists():
        target.parent.mkdir(exist_ok=True)
        shutil.copyfile(glyph_file, target)
    safe_unpickle(target)  # той самий набір класів numpy, що й у VATr; інакше впаде тут
    with in_repo(repo), quiet():
        from data_loader.loader import ContentData, letters
        cd = ContentData()  # loader.py:254 — читає data/unifont.pickle
        index2letter = {v: k for k, v in cd.letter2index.items()}

        def encode(text):  # test.py:96 → loader.py:259
            cd.get_content(text)
            return [cd.letter2index[c] for c in text]

        decode = lambda ids: "".join(index2letter[i] for i in ids)
        lvl1 = run_chars(entries, _roundtrip(encode, decode), repo)
        lvl2 = run_strings(strings, lambda t: (" ".join(decode(encode(w)) for w in t.split()), None), repo)
    glyphs, n = _vatr_glyphs(target, entries)
    return {
        "encoder": f"data_loader/loader.py:25 letters: {len(letters)} символів (IAM)",
        "paths": {"test.py → ContentData.get_content": {"level1": lvl1, "level2": lvl2}},
        "glyphs": {"file": "English_data.zip: unifont.pickle", "n_symbols": n, "entries": glyphs,
                   "note": "Файл байт у байт збігається з files/unifont.pickle VATr"},
    }


def probe_wordstylist(entries, strings):
    repo = REPOS / "WordStylist"
    with in_repo(repo), quiet():
        import train  # на рівні модуля лише будує словник (train.py:59) і пише letter2index.json

        def encode(text):  # sampling.py → Diffusion.sampling → train.py:189 label_padding
            return train.label_padding(text, train.num_tokens)

        def decode(ids):  # label_padding доповнює PAD_TOKEN до OUTPUT_MAX_LEN
            body = [i for i in ids if i != train.tokens["PAD_TOKEN"]]
            return "".join(train.index2letter[i - train.num_tokens] for i in body)

        lvl1 = run_chars(entries, _roundtrip(encode, decode), repo)
        lvl2 = run_strings(strings, lambda t: (" ".join(decode(encode(w)) for w in t.split()), None), repo)
    return {
        "encoder": f"train.py:19 c_classes: {len(train.c_classes)} латинських літер",
        "paths": {"sampling.py → train.py:189 label_padding": {"level1": lvl1, "level2": lvl2}},
    }


PROBES = {
    "GANwriting": probe_ganwriting,
    "HiGAN+": probe_higan,
    "HWT": probe_hwt,
    "VATr": probe_vatr,
    "VATr++": probe_vatrpp,
    "One-DM": probe_onedm,
    "WordStylist": probe_wordstylist,
}

In [ ]:
%%writefile scripts/level12/probes_tokenizer.py
"""Зонди методів із відкритим токенізатором: DiffusionPen, українська DiffusionPen (CANINE),
Emuru, Eruku (ByT5). Токенізатори завантажуються на ревізіях із versions/pinned.json.
"""

import ast
import types

import numpy as np

from common import Captured, REPOS, Site, in_repo, pinned, quiet, run_chars, run_strings


def _hf_revision(model_id):
    return pinned(model_id)["sha"]


def _canine():
    from transformers import CanineTokenizer
    return CanineTokenizer.from_pretrained("google/canine-c", revision=_hf_revision("google/canine-c"))


def _byt5():
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained("google/byt5-small", revision=_hf_revision("google/byt5-small"))


def _tok_roundtrip(tok, call, max_len=None):
    def fn(ch):
        ids = call(ch)
        back = tok.decode(ids, skip_special_tokens=True)
        status = "ok" if back == ch else "lossy"
        return status, back, None
    return fn


def probe_diffusionpen(entries, strings):
    repo = REPOS / "DiffusionPen"
    tok = _canine()
    site = Site(repo, "train.py", 263, 'tokenizer(text_features, padding="max_length", truncation=True, return_tensors="pt", max_length=40)')

    def encode(text):  # train.py:263 — ті самі параметри, що в Diffusion.sampling
        return tok(text, padding="max_length", truncation=True, return_tensors="pt", max_length=40).input_ids[0].tolist()

    lvl1 = run_chars(entries, _tok_roundtrip(tok, encode), repo)
    # Режим paragraph (train.py:757): lines.strip().split(' '), кожне слово окремо в sampling.
    lvl2 = run_strings(strings, lambda t: (" ".join(tok.decode(encode(w), skip_special_tokens=True)
                                                     for w in t.strip().split(" ")), site), repo)
    return {
        "encoder": "CANINE (google/canine-c): кодові точки Unicode; max_length=40 разом зі службовими токенами",
        "paths": {"train.py --train_mode sampling → Diffusion.sampling → CanineTokenizer": {"level1": lvl1, "level2": lvl2}},
        "notes": ["Слова, довші за 38 кодових точок, обрізаються (truncation=True, train.py:263)"],
    }


def _load_space_function(path, name, namespace):
    """Виконує одну функцію з файлу як є, не імпортуючи сам файл (app.py при імпорті вантажить моделі)."""
    tree = ast.parse(path.read_text(encoding="utf-8"))
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == name)
    node.decorator_list = []  # @spaces.GPU лише виділяє GPU в Hugging Face Spaces
    exec(compile(ast.Module(body=[node], type_ignores=[]), str(path), "exec"), namespace)
    return namespace[name], node.lineno


def probe_ukr_diffusionpen(entries, strings):
    repo = REPOS / "UkrDiffusionPen-space"
    tok = _canine()
    with in_repo(repo), quiet():
        import generate_sentence as gs
        captured = {}

        def generate_single_word(word, tokenizer, text_max_len, **_):
            # generate_sentence.py:1216 encode_text_context: ті самі параметри токенізатора
            ids = tokenizer(word, padding="max_length", truncation=True, return_tensors="pt",
                            max_length=text_max_len).input_ids[0].tolist()
            captured.setdefault("words", []).append(tokenizer.decode(ids, skip_special_tokens=True))
            return gs.Image.new("L", (8, 64), 255)

        real_sample = gs.sample_punctuation

        def sample_punctuation(ch, *a, **k):
            arr = real_sample(ch, *a, **k)
            captured.setdefault("punct", []).append((ch, arr is not None))
            return arr

        def compose_sentence_geometry(**kw):
            raise Captured(kw)

        ns = {
            **{k: getattr(gs, k) for k in ("PUNCTUATION", "split_word_for_generation", "crop_whitespace")},
            "np": np, "torch": __import__("torch"), "random": __import__("random"), "time": __import__("time"),
            "sample_punctuation": sample_punctuation, "generate_single_word": generate_single_word,
            "compose_sentence_geometry": compose_sentence_geometry,
            "normalize_ink_brightness": lambda imgs: imgs,
            "MODEL": {"unet": types.SimpleNamespace(to=lambda d: None), "vae": types.SimpleNamespace(to=lambda d: None),
                      "style_extractor": types.SimpleNamespace(to=lambda d: None),
                      "tokenizer": tok, "noise_scheduler": None},
            "_random_writer": lambda lucky=False: "0000", "WRITER_ID_MAP": {"0000": 0},
            "_load_writer_style_ref": lambda w, d: None, "_load_recent_table": lambda: None,
            "PUNCT_BANK_DIR": str(repo / "punct_bank"), "IMG_HEIGHT": 64, "IMG_WIDTH": 256,
            "TEXT_MAX_LEN": 40, "CANVAS_HEIGHT": 104,
        }
        generate, lineno = _load_space_function(repo / "app.py", "generate", ns)
        site_bank = Site(repo, "app.py", 324, "word_images.append(ch_arr if ch_arr is not None else np.full((IMG_HEIGHT, 6), 255")

        def app_path(text):
            """Справжня app.generate по одному слову; результат — те, що реально потрапляє на зображення."""
            out_words = []
            for word in text.split():
                captured.clear()
                try:
                    generate(word, 0)
                except Captured as c:
                    kw = c.payload
                    punct_ok = iter(captured.get("punct", []))
                    gen_words = iter(captured.get("words", []))
                    parts = []
                    for w, is_p in zip(kw["expanded_words"], kw["punct_flags"]):
                        if is_p:
                            ch, has_img = next(punct_ok)
                            parts.append(ch if has_img else "")
                        else:
                            parts.append(next(gen_words))
                    out_words.append("".join(parts))
                else:
                    out_words.append("")
            return " ".join(out_words), site_bank

        def char(ch):
            out, s = app_path(f"а{ch}")  # знак у кінці слова: так його обробляє app.generate
            got = out[1:] if out.startswith("а") else out
            return ("ok" if got == ch else "lost" if got == "" else "lossy"), got, (s if got != ch else None)

        lvl1 = run_chars(entries, _tok_roundtrip(tok, lambda ch: tok(ch).input_ids), repo)
        lvl2_chars = run_chars(entries, char, repo)
        lvl2 = run_strings(strings, app_path, repo)
        bank = sorted(p.name for p in (repo / "punct_bank").iterdir() if p.is_dir())
    return {
        "encoder": "CANINE (google/canine-c); розділові знаки в кінці слова — із банку зображень punct_bank",
        "paths": {
            "CanineTokenizer (кодування окремого символу)": {"level1": lvl1},
            f"app.py:{lineno} generate → punct_bank / CANINE": {"level1": lvl2_chars, "level2": lvl2},
        },
        "notes": [
            f"Теки punct_bank: {', '.join(bank)}; _PUNCT_SUBDIR (generate_sentence.py:902) зіставляє з ними , . ? ! : ; -",
            "Знак із PUNCTUATION (generate_sentence.py:40) у кінці слова без теки в банку замінюється порожнім полем 6 пікс.",
            "app.generate виконано з файлу app.py без імпорту модуля (при імпорті він вантажить моделі й будує інтерфейс)",
        ],
    }


def probe_emuru(entries, strings):
    repo = REPOS / "Emuru"
    tok = _byt5()
    site = Site(repo, "models/emuru.py", 154, "texts = [style_text + ' ' + gen_text]")

    def encode(text):  # models/emuru.py:154 і :228
        return tok([text], return_tensors="pt", padding=True).input_ids[0].tolist()

    lvl1 = run_chars(entries, _tok_roundtrip(tok, encode), repo)
    lvl2 = run_strings(strings, lambda t: (tok.decode(encode("style" + " " + t), skip_special_tokens=True)[len("style "):], site), repo)
    utf8 = {e["id"]: len(e["char"].encode("utf-8")) for e in entries}
    return {
        "encoder": "ByT5 (google/byt5-small): байти UTF-8",
        "paths": {"models/emuru.py generate → tokenizer(style_text + ' ' + gen_text)": {"level1": lvl1, "level2": lvl2}},
        "utf8_bytes": utf8,
        "notes": ["Кирилична літера — 2 байти UTF-8, ꙋ і ʼ — 2–3: кодується без втрат, але як послідовності байтів, яких у латинському навчанні не було"],
    }


def probe_eruku(entries, strings, repo_name="Eruku"):
    repo = REPOS / repo_name
    tok = _byt5()
    tok.add_tokens(["<sog>"])  # eruku_continuous_inf.py: self.tokenizer.add_tokens(["<sog>"])
    Site.find(repo, "eruku_continuous_inf.py", 'self.tokenizer.add_tokens(["<sog>"])')
    site = Site.find(repo, "eruku_continuous_inf.py", 'for style, gen in zip(style_text,gen_text)')

    def encode(text):
        return tok([text], padding=True, return_tensors="pt").input_ids[0].tolist()

    def gen_path(text):  # eruku_continuous_inf.py:365 — f"{style}<sog>{gen}"
        decoded = tok.decode(encode(f"style<sog>{text}"), skip_special_tokens=False)
        return decoded.split("<sog>", 1)[1].replace("</s>", ""), site

    lvl1 = run_chars(entries, _tok_roundtrip(tok, encode), repo)
    lvl2 = run_strings(strings, gen_path, repo)
    return {
        "encoder": "ByT5 (google/byt5-small) + токен <sog>",
        "paths": {"eruku_continuous_inf.py generate → tokenizer(f'{style}<sog>{gen}')": {"level1": lvl1, "level2": lvl2}},
    }


PROBES = {
    "DiffusionPen": probe_diffusionpen,
    "Українська DiffusionPen": probe_ukr_diffusionpen,
    "Emuru": probe_emuru,
    "Eruku": probe_eruku,
    "Eruku (aimagelab, версія пропозиції)": lambda e, s: probe_eruku(e, s, "Eruku-aimagelab"),
}

In [ ]:
%%writefile scripts/level12/fetch_sources.py
"""Завантажує код методів на зафіксованих ревізіях (versions/pinned.json) у work/repos.

- git-репозиторії методів: лише потрібний коміт (git fetch --depth 1);
- Space української DiffusionPen: код і по 2 зображення з кожної теки punct_bank;
- файли гліфів unifont.pickle моделі VATr++ з Hugging Face і One-DM з архіву даних
  на Google Drive (з архіву читається лише цей файл).
Ваги моделей не завантажуються: для рівнів 1–2 вони не потрібні.
Токенізатори CANINE і ByT5 завантажує transformers на зафіксованих ревізіях під час зондування.

Запуск: python3 scripts/level12/fetch_sources.py
"""

import hashlib
import io
import re
import zipfile
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import REPOS, pinned  # noqa: E402

import json  # noqa: E402

PINNED = json.loads((Path(__file__).resolve().parents[2] / "versions" / "pinned.json").read_text(encoding="utf-8"))


def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "-", name.replace("+", "plus")).strip("-")


def fetch_git(row):
    dest = REPOS / slug(row["name"])
    sha = row["commit"]
    if (dest / ".git").exists() and subprocess.run(
            ["git", "-C", dest, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() == sha:
        return dest, "вже є"
    dest.mkdir(parents=True, exist_ok=True)
    run = lambda *a: subprocess.run(["git", "-C", dest, *a], check=True, capture_output=True)
    if not (dest / ".git").exists():
        run("init", "-q")
        run("remote", "add", "origin", f"https://github.com/{row['repo']}.git")
    run("fetch", "-q", "--depth", "1", "origin", sha)
    run("-c", "advice.detachedHead=false", "checkout", "-q", "FETCH_HEAD")
    return dest, "завантажено"


def hf_file(repo, kind, sha, path, dest, lfs_sha256=None):
    prefix = "spaces/" if kind == "space" else ""
    url = f"https://huggingface.co/{prefix}{repo}/resolve/{sha}/{path}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(4):
        if dest.exists():
            break
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                data = r.read()
            dest.write_bytes(data)
        except OSError:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)
    if lfs_sha256:
        got = hashlib.sha256(dest.read_bytes()).hexdigest()
        if got != lfs_sha256:
            raise RuntimeError(f"{path}: SHA-256 {got} ≠ {lfs_sha256}")


def fetch_space():
    row = next(h for h in PINNED["huggingface"]
               if h["id"] == "kdonitz/diffusionpen-ukrainian" and h["kind"] == "space")
    dest = REPOS / "UkrDiffusionPen-space"
    bank = defaultdict(list)
    for f in row["files"]:
        p = f["path"]
        if p.endswith(".py") or p in ("README.md", "requirements.txt"):
            hf_file(row["id"], "space", row["sha"], p, dest / p, f["lfs_sha256"])
        elif p.startswith("punct_bank/") and p.endswith(".png"):
            bank[p.split("/")[1]].append(f)
    for files in bank.values():
        for f in sorted(files, key=lambda x: x["path"])[:2]:
            hf_file(row["id"], "space", row["sha"], f["path"], dest / f["path"], f["lfs_sha256"])
    (dest / "PINNED_SHA").write_text(row["sha"] + "\n")
    return dest, f"{row['sha'][:12]}, punct_bank: {', '.join(sorted(bank))}"


def fetch_vatrpp_glyphs():
    row = next(h for h in PINNED["huggingface"] if h["id"] == "blowing-up-groundhogs/vatrpp")
    f = next(x for x in row["files"] if x["path"] == "files/unifont.pickle")
    dest = REPOS / "vatrpp-hf" / "files" / "unifont.pickle"
    hf_file(row["id"], "model", row["sha"], f["path"], dest, f["lfs_sha256"])
    return dest, f"{row['sha'][:12]}, {f['size'] / 2**20:.1f} МБ"


class RangeFile(io.RawIOBase):
    """Файл на HTTP-сервері, який читаємо діапазонами байтів (Range): zipfile бере лише потрібне."""

    def __init__(self, url):
        self.url, self.pos = url, 0
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req, timeout=60) as r:
            self.size = int(r.headers["Content-Length"])

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        self.pos = {io.SEEK_SET: offset, io.SEEK_CUR: self.pos + offset, io.SEEK_END: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        end = self.size if n is None or n < 0 else min(self.size, self.pos + n)
        if self.pos >= end:
            return b""
        req = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end - 1}"})
        with urllib.request.urlopen(req, timeout=120) as r:
            data = r.read()
        self.pos += len(data)
        return data


# One-DM: файл гліфів лежить лише в архіві даних на Google Drive (README, розділ Datasets)
ONEDM_ZIP = "https://drive.usercontent.google.com/download?id=10BMCU38cBJfLebzrS3LVl47RtBrRdxOz&export=download&confirm=t"


def fetch_onedm_glyphs():
    dest = REPOS / "onedm-data" / "unifont.pickle"
    info = dest.with_suffix(".source.json")
    if not dest.exists():
        with zipfile.ZipFile(RangeFile(ONEDM_ZIP)) as z:
            member = next(n for n in z.namelist() if n.endswith("unifont.pickle"))
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(z.read(member))
            info.write_text(json.dumps({
                "archive": "English_data.zip (Google Drive 10BMCU38cBJfLebzrS3LVl47RtBrRdxOz)",
                "member": member, "sha256": hashlib.sha256(dest.read_bytes()).hexdigest(),
            }, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    meta = json.loads(info.read_text(encoding="utf-8"))
    return dest, f"{meta['member']}, sha256 {meta['sha256'][:12]}"


def main():
    REPOS.mkdir(parents=True, exist_ok=True)
    for row in PINNED["git"]:
        if row["role"] in ("method", "related", "synthesis_tool") and row["status"] == "ok":
            dest, msg = fetch_git(row)
            print(f"{row['name']:<38} {row['commit'][:12]} {msg}", flush=True)
    for fn, label in ((fetch_space, "Space української DiffusionPen"), (fetch_vatrpp_glyphs, "VATr++ unifont.pickle"),
                      (fetch_onedm_glyphs, "One-DM unifont.pickle")):
        dest, msg = fn()
        print(f"{label:<38} {msg}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/level12/run_level12.py
"""Рівні 1–2 для всіх методів: запускає зонди й зводить таблицю «символ × метод».

Спершу: python3 scripts/level12/fetch_sources.py
Запуск:  python3 scripts/level12/run_level12.py [назва методу ...]
Результати: results/level12/<метод>.json, symbol_x_method.csv, summary.md
"""

import csv
import json
import sys
import traceback
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import RESULTS, inventory, pinned, save, test_strings  # noqa: E402
import probes_alphabet  # noqa: E402
import probes_tokenizer  # noqa: E402

PROBES = {**probes_alphabet.PROBES, **probes_tokenizer.PROBES}
SOURCE = {"Українська DiffusionPen": "kdonitz/diffusionpen-ukrainian"}
GROUPS = ("alphabet", "apostrophe", "punctuation", "historical")


def version_of(method):
    if method in SOURCE:
        row = next(h for h in pinned_all() if h.get("id") == SOURCE[method] and h["kind"] == "space")
        return {"source": f"hf space {row['id']}", "revision": row["sha"]}
    row = pinned(method)
    return {"source": f"github {row['repo']}", "revision": row["commit"]}


def environment():
    import platform
    from importlib.metadata import PackageNotFoundError, version
    env = {"python": platform.python_version(), "platform": platform.platform()}
    for pkg in ("torch", "transformers", "numpy"):
        try:
            env[pkg] = version(pkg)
        except PackageNotFoundError:
            env[pkg] = None
    return env


def pinned_all():
    from common import ROOT
    return json.loads((ROOT / "versions" / "pinned.json").read_text(encoding="utf-8"))["huggingface"]


def main(selected):
    inv, entries = inventory()
    strings = test_strings(inv)
    table = {}
    for method, probe in PROBES.items():
        if selected and method not in selected:
            continue
        try:
            payload = probe(entries, strings)
            payload["status"] = "ok"
        except Exception as exc:  # noqa: BLE001 — зонд, що впав, теж результат
            payload = {"status": "probe_failed", "error": f"{type(exc).__name__}: {exc}",
                       "traceback": traceback.format_exc()[-2000:]}
        payload.update(method=method, inventory_version=inv["inventory_version"], environment=environment(),
                       checked_at=datetime.now(timezone.utc).isoformat(timespec="seconds"), **version_of(method))
        save(method, payload)
        report(method, payload, entries)
    # Зведення — з усіх збережених результатів, тож методи можна запускати й окремо
    for method in PROBES:
        path = RESULTS / f"{method}.json"
        if path.exists():
            table[method] = json.loads(path.read_text(encoding="utf-8"))
    write_table(table, entries)


def report(method, payload, entries):
    if payload["status"] != "ok":
        print(f"\n== {method}: ЗОНД НЕ ВИКОНАНО — {payload['error']}")
        return
    print(f"\n== {method} [{payload['revision'][:12]}] — {payload['encoder']}")
    by_id = {e["id"]: e for e in entries}
    for path, res in payload["paths"].items():
        if "level1" in res:
            c = Counter((by_id[i]["group"], r["status"]) for i, r in res["level1"].items())
            parts = []
            for g in GROUPS:
                n = sum(v for (gg, _), v in c.items() if gg == g)
                ok = c.get((g, "ok"), 0)
                parts.append(f"{g} {ok}/{n}")
            sites = Counter(r["where"]["ref"] for r in res["level1"].values() if r.get("where"))
            print(f"  {path}\n    рівень 1: {', '.join(parts)}" + (f"; втрати: {dict(sites)}" if sites else ""))
        if "level2" in res:
            for k, r in res["level2"].items():
                if r["status"] != "ok":
                    detail = r.get("error") or f"втрачено {r['lost']} змінено {r['changed']}"
                    where = f" @ {r['where']['ref']}" if r.get("where") else ""
                    print(f"    рівень 2 [{k}]: {r['status']} — {detail[:150]}{where}")


def write_table(table, entries):
    cols = []
    for method, p in table.items():
        if p["status"] != "ok":
            cols.append((method, None, None))
            continue
        for path, res in p["paths"].items():
            if "level1" in res:
                cols.append((method, path, res["level1"]))
        if "glyphs" in p and "entries" in p["glyphs"]:
            cols.append((method, "гліфи " + p["glyphs"]["file"], p["glyphs"]["entries"]))
    RESULTS.mkdir(parents=True, exist_ok=True)
    with open(RESULTS / "symbol_x_method.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id", "char", "codepoints", "group"] + [f"{m} | {p}" if p else m for m, p, _ in cols])
        for e in entries:
            row = [e["id"], e["char"], " ".join(e["codepoints"]), e["group"]]
            for _, _, res in cols:
                if res is None:
                    row.append("probe_failed")
                else:
                    r = res[e["id"]]
                    row.append(r["status"] + (f" @ {r['where']['ref']}" if r.get("where") else ""))
            w.writerow(row)

    lines = ["# Рівні 1–2: зведення", "",
             "Частка символів інвентарю зі статусом ok за групами. Повні дані — symbol_x_method.csv і <метод>.json.", "",
             "| Метод | Шлях | " + " | ".join(GROUPS) + " | Де губиться |", "|---|---|" + "---|" * (len(GROUPS) + 1)]
    for m, p, res in cols:
        if res is None:
            lines.append(f"| {m} | — | " + " | ".join("зонд не виконано" for _ in GROUPS) + " | |")
            continue
        cells = []
        for g in GROUPS:
            ids = [e["id"] for e in entries if e["group"] == g]
            ok = sum(res[i]["status"] == "ok" for i in ids)
            cells.append(f"{ok}/{len(ids)}")
        sites = sorted({r["where"]["ref"] for r in res.values() if r.get("where")})
        lines.append(f"| {m} | {p} | " + " | ".join(cells) + f" | {', '.join(sites)} |")
    (RESULTS / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"\n{RESULTS / 'symbol_x_method.csv'}\n{RESULTS / 'summary.md'}")


if __name__ == "__main__":
    main(sys.argv[1:])

In [ ]:
# Код методів на зафіксованих ревізіях (потрібен і для кроку 7)
!python3 scripts/level12/fetch_sources.py

In [ ]:
# Необов'язково: зонди рівнів 1–2 (результати вже є локально; на CPU кілька хвилин)
# !python3 scripts/level12/run_level12.py 2>&1 | grep -v "Warning\|warn"

## Крок 4. На чому навчено кожен checkpoint

Дані зібрано вручну з карток моделей, README і статей (кожне твердження має джерело) у `checkpoints/training_data.json`; скрипт робить із них таблицю `checkpoints/training_data.md`.

In [ ]:
%%writefile checkpoints/training_data.json
{
  "about": "Крок 4: на чому навчено кожен checkpoint. Лише з карток моделей, README і статей; кожне твердження має джерело. Переглянуто 23.10.2026.",
  "methods": [
    {
      "method": "GANwriting",
      "checkpoint": "Не опубліковано: README пропонує навчити модель самостійно (run_train_scratch.sh, run_train_pretrain.sh)",
      "datasets": [
        "IAM, слова: 62 857 зображень, 500 авторів; 160 авторів відкладено для тесту"
      ],
      "content_text": "22 500 англійських слів із корпусу Brown",
      "language": "англійська, латиниця",
      "stages": [
        "Генератор, дискримінатор, класифікатор автора й розпізнавач навчаються разом з нуля (попереднього навчання немає)"
      ],
      "style_refs": "50 зображень слів: NUM_CHANNEL = 50 (load_data.py:15, значення 15 закоментоване), тестовий скрипт бере imgs[:50] (рядок 109)",
      "sources": [
        "Kang et al., ECCV 2020, arXiv:2003.02567, розд. 3–4",
        "README репозиторію"
      ]
    },
    {
      "method": "HiGAN+",
      "checkpoint": "HiGAN+/pretrained/deploy_HiGAN+.pth у репозиторії (коміт 5d37de7); допоміжні ocr_iam_new.pth, wid_iam_new.pth",
      "datasets": [
        "IAM, слова (configs/gan_iam.yml; оброблені h5py-файли з релізу dataset)"
      ],
      "language": "англійська, латиниця",
      "stages": [
        "Розпізнавач (ocr_iam.yml) і класифікатор автора (wid_iam.yml) навчаються окремо на IAM",
        "GAN (gan_iam.yml) на IAM з цими допоміжними мережами"
      ],
      "style_refs": "не з'ясовано: eval_style бере зразки з тестового набору (networks/model.py:445–462)",
      "sources": [
        "README і configs репозиторію"
      ],
      "gaps": [
        "Статтю (Gan et al., ACM TOG 2022) не переглянуто: опис етапів — за конфігураціями"
      ]
    },
    {
      "method": "HWT",
      "checkpoint": "Google Drive (README): ваги для IAM і CVL",
      "datasets": [
        "IAM, слова: 62 857 англійських слів, 500 авторів; навчання — 340 авторів, тест — 160",
        "CVL (окремий checkpoint)"
      ],
      "language": "англійська, латиниця",
      "stages": [
        "Одне навчання трансформера з GAN-втратами, розпізнавачем і класифікатором автора на IAM"
      ],
      "style_refs": "P = 15 зображень слів",
      "sources": [
        "Bhunia et al., ICCV 2021, arXiv:2104.03964, розд. 4",
        "README"
      ]
    },
    {
      "method": "VATr",
      "checkpoint": "Google Drive (README): ваги для IAM; також blowing-up-groundhogs/vatrpp: vatr/",
      "datasets": [
        "Попереднє навчання стильового енкодера: синтетика — 10 400 англійських слів × 10 400 каліграфічних шрифтів, понад 100 млн зразків",
        "IAM, слова: 340 авторів навчання, 160 тест; алфавіт 79 символів"
      ],
      "language": "англійська, латиниця",
      "stages": [
        "1) Згорткова основа стильового енкодера: класифікація шрифту на синтетиці (Cross-Entropy)",
        "2) Генератор на IAM"
      ],
      "style_refs": "P = 15 зображень слів",
      "text_representation": "Гліфи GNU Unifont як запити трансформера",
      "sources": [
        "Pippi et al., CVPR 2023, arXiv:2303.15269, розд. 3.1.1 і 4"
      ]
    },
    {
      "method": "VATr++",
      "checkpoint": "Google Drive (README); blowing-up-groundhogs/vatrpp (ревізія fb73be2)",
      "datasets": [
        "Попереднє навчання стильового енкодера: та сама синтетика, що у VATr (понад 100 млн зразків, 10 400 слів × 10 400 шрифтів)",
        "IAM, слова, з детермінованим поділом; розділові знаки приєднано до слів"
      ],
      "language": "англійська, латиниця",
      "stages": [
        "1) Стильовий енкодер: класифікація шрифту на синтетиці",
        "2) Генератор на IAM з аугментацією тексту (вирівнювання розподілу рідкісних символів) і стилю; HTR і класифікатор автора навчаються на реальних зображеннях IAM"
      ],
      "style_refs": "P = 15 зображень слів",
      "sources": [
        "Vanherle et al., 2024, arXiv:2402.10798, розд. III–IV",
        "картка blowing-up-groundhogs/vatrpp"
      ]
    },
    {
      "method": "WordStylist",
      "checkpoint": "Google Drive (README)",
      "datasets": [
        "IAM, слова: поділ Aachen, слова з 2–7 символів; 339 стилів авторів, ≈45 тис. слів"
      ],
      "language": "англійська, латиниця",
      "stages": [
        "Латентна дифузія (VAE Stable Diffusion, попередньо навчений, замороженний) на IAM, 1000 епох"
      ],
      "style_refs": "мітка автора (339 класів), а не зразки: нові стилі без перенавчання неможливі",
      "sources": [
        "Nikolaidou et al., ICDAR 2023, arXiv:2303.16576, розд. 3–4"
      ]
    },
    {
      "method": "One-DM",
      "checkpoint": "Google Drive / Baidu / wisemodel (README): Pretrained One-DM, OCR і ResNet18",
      "datasets": [
        "IAM, слова: 339 авторів навчання, 161 тест",
        "CVL, лише англійська частина: 84 514 слів, 283 автори навчання"
      ],
      "language": "англійська, латиниця; експерименти з китайською (ICDAR-2013) і японською (UP_Kuchibue) описано в статті, ваги для них у README не вказано",
      "stages": [
        "1) 700 епох дифузії з classifier-free guidance",
        "2) Донавчання 4500 ітерацій з розпізнавачем (CTC)"
      ],
      "style_refs": "1 зображення",
      "text_representation": "Гліфи Unifont для символів алфавіту IAM",
      "sources": [
        "Dai et al., ECCV 2024, arXiv:2409.04004, розд. 4.1 і 4.4",
        "README"
      ]
    },
    {
      "method": "DiffusionPen",
      "checkpoint": "konnik/DiffusionPen (ревізія 5636a68): diffusionpen_iam_model_path, style_models",
      "datasets": [
        "IAM, слова: 339 авторів для стильового енкодера й дифузії, 160 для оцінки",
        "GNHK — лише якісні приклади"
      ],
      "language": "англійська, латиниця",
      "stages": [
        "1) Стильовий енкодер MobileNetV2: класифікація + metric learning на IAM, 20 епох",
        "2) Латентна дифузія (VAE SD 1.5) разом із текстовим енкодером CANINE-C"
      ],
      "style_refs": "5 зображень слів",
      "text_encoder_pretraining": "CANINE-C: Вікіпедія 104 мовами (картка google/canine-c); у DiffusionPen текстовий енкодер навчається разом із дифузією",
      "sources": [
        "Nikolaidou et al., ECCV 2024, arXiv:2409.06065, розд. 3–4",
        "картка konnik/DiffusionPen"
      ]
    },
    {
      "method": "Українська DiffusionPen",
      "checkpoint": "kdonitz/diffusionpen-ukrainian (ревізія a96778d): ema_ckpt.pt, mixed_ukr_mobilenetv2_100.pth; картки моделі немає",
      "datasets": [
        "126 177 зображень слів від 308 авторів, побудованих з UkrHandwritten (37 111 рядків, 331 автор)"
      ],
      "language": "сучасна українська, кирилиця",
      "data_filters": [
        "Вилучено мітки з латиницею, лише розділовими знаками чи цифрами",
        "Вилучено вирізки, чия мітка закінчувалася комою",
        "Перевірка кириличним TrOCR (поріг 0,4 зі ступінчастим послабленням для коротких слів)",
        "Рідкісні літери ф, ґ, Щ, Є, Ц, ї продубльовано 2–5 разів"
      ],
      "stages": [
        "Архітектура DiffusionPen без змін (лише num_res_blocks = 2); стильовий енкодер MobileNetV2 з triplet loss, заморожений під час дифузії",
        "Дифузія 200 епох, VAE SD 1.5 заморожений, p_drop тексту 0,2"
      ],
      "style_refs": "5 зображень слів + мітка автора",
      "punctuation": "Кома, крапка, дефіс вставляються з банку 500 реальних знаків, а не генеруються",
      "historical_letters": "У навчальних даних — сучасний правопис; ы, ъ, ѣ та інші історичні літери не засвідчені",
      "sources": [
        "Ahitoliev, Berezin, 2026, arXiv:2605.27487, розд. 3.1–3.4",
        "README Space kdonitz/diffusionpen-ukrainian"
      ],
      "gaps": [
        "Невідомо, яким символом апострофа записано мітки (U+02BC, U+2019 чи U+0027)",
        "Чи ваги на HF — ті самі, що в статті, не підтверджено"
      ]
    },
    {
      "method": "Emuru",
      "checkpoint": "blowing-up-groundhogs/emuru (ревізія c75b78d) і emuru_vae (139f13a)",
      "datasets": [
        "Лише синтетика: 2,2 млн зображень рядків англійського тексту (слова з корпусів NLTK) у понад 100 тис. шрифтів (font-square-v2)"
      ],
      "language": "англійська, латиниця; IAM, CVL, RIMES і Karaoke (англ., фр., нім., італ.) — лише для оцінки без донавчання",
      "stages": [
        "1) VAE на синтетиці (60 000 ітерацій; допоміжні HTR і класифікатор автора, навчені на тій самій синтетиці)",
        "2) Авторегресійний T5 при замороженому VAE: спершу рядки з 4–7 слів, потім донавчання на рядках з 1–32 слів"
      ],
      "style_refs": "1 зображення рядка + його транскрипція",
      "text_encoder_pretraining": "Токенізатор ByT5 (байти UTF-8); ByT5 попередньо навчено на mC4 (картка google/byt5-small)",
      "sources": [
        "Pippi et al., CVPR 2025, arXiv:2503.17074, розд. 3.1–3.3 і 4",
        "картки emuru, emuru_vae"
      ]
    },
    {
      "method": "Eruku",
      "checkpoint": "blowing-up-groundhogs/eruku (ревізія 13f4f75)",
      "datasets": [
        "Стаття: лише синтетика — понад 100 тис. шрифтів, англійські й випадкові слова; 23 млн зразків для 1-го етапу, 10 млн для 2-го (font-square-pretrain-20M)"
      ],
      "language": "англійська, латиниця (картка: «trained primarily on English»)",
      "stages": [
        "1) 65 000 ітерацій: стильове зображення 2–3 слова, ціль 2–3 слова, з навчанням безумовної генерації для CFG",
        "2) 5000 ітерацій: без тексту стилю з імовірністю p_drop, ціль 1–32 слова; OCR-донавчання у фінальній моделі не використано",
        "VAE — ваги Emuru"
      ],
      "style_refs": "1 зображення; транскрипція необов'язкова",
      "text_encoder_pretraining": "Токенізатор ByT5 + токен <sog>",
      "sources": [
        "Zaccagnino et al., WACV 2026, arXiv:2510.23240, розд. 4.1–4.2 і 5",
        "картка blowing-up-groundhogs/eruku"
      ],
      "conflicts": [
        "Картка моделі перелічує «Training datasets: IAM, CVL, RIMES, FontSquare», а стаття — що модель навчено лише на синтетиці, а IAM, CVL і RIMES використано для оцінки без донавчання. Пропозиція спиралася на картку"
      ]
    }
  ]
}

In [ ]:
%%writefile scripts/render_training_data.py
"""Таблиця кроку 4 з checkpoints/training_data.json → checkpoints/training_data.md."""

import json
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
SRC = ROOT / "checkpoints" / "training_data.json"
OUT = SRC.with_suffix(".md")


def cell(value):
    if isinstance(value, list):
        return "<br>".join(value)
    return (value or "—").replace("|", "\\|")


def main():
    data = json.loads(SRC.read_text(encoding="utf-8"))
    lines = ["# Крок 4. На чому навчено кожен checkpoint", "", data["about"], "",
             "| Метод | Checkpoint | Навчальні дані | Мова | Етапи навчання | Зразки стилю | Джерела |",
             "|---|---|---|---|---|---|---|"]
    notes = []
    for m in data["methods"]:
        lines.append("| " + " | ".join(cell(m.get(k)) for k in
                     ("method", "checkpoint", "datasets", "language", "stages", "style_refs", "sources")) + " |")
        for key, label in (("conflicts", "Розбіжність"), ("gaps", "Не з'ясовано"), ("punctuation", "Розділові знаки"),
                           ("historical_letters", "Історичні літери"), ("data_filters", "Фільтри даних"),
                           ("text_encoder_pretraining", "Текстовий енкодер"), ("text_representation", "Подання тексту"),
                           ("content_text", "Текст для генерації")):
            if m.get(key):
                notes.append(f"- **{m['method']} — {label}:** {cell(m[key]).replace('<br>', '; ')}")
    lines += ["", "## Примітки", ""] + notes
    OUT.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(OUT.relative_to(ROOT))


if __name__ == "__main__":
    main()

In [ ]:
!python3 scripts/render_training_data.py

## Крок 5. Аудит шрифтів

Відбір рукописних шрифтів Google Fonts з кирилицею на зафіксованому коміті; Kobzar KS (архів Wayback, звірка SHA-256); шрифти інструментів синтетики (ukrainian-handwriting-synth, doc_gen). Для кожного символу інвентарю: cmap (fontTools; для Kobzar KS — лише HarfBuzz, бо ліцензія забороняє розбір шрифту), .notdef у шейпінгу HarfBuzz, рендеринг Pillow з libraqm. Окремо — чи застосовуються OpenType-варіанти Kobzar KS. Результати вже є локально; клітинка запуску необов'язкова.

In [ ]:
%%writefile scripts/fonts/select_fonts.py
"""Крок 5а: відбір рукописних шрифтів Google Fonts з кирилицею на зафіксованому коміті.

Критерій пропозиції: category = HANDWRITING і subsets містить cyrillic; окремо — сімейства,
у яких HANDWRITING є лише в полі classifications. Завантажує з google/fonts тільки
METADATA.pb (sparse checkout), а потім — файли шрифтів відібраних сімейств.

Запуск: python3 scripts/fonts/select_fonts.py
Результат: results/fonts/selection.json
"""

import json
import re
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "scripts" / "level12"))
from common import WORK, pinned  # noqa: E402

GF = WORK / "repos" / "google-fonts"
OUT = ROOT / "results" / "fonts" / "selection.json"


def git(*args):
    subprocess.run(["git", "-C", GF, *args], check=True, capture_output=True)


def sparse_checkout(patterns):
    sha = pinned("google/fonts")["commit"]
    GF.mkdir(parents=True, exist_ok=True)
    if not (GF / ".git").exists():
        git("init", "-q")
        git("remote", "add", "origin", "https://github.com/google/fonts.git")
    git("sparse-checkout", "set", "--no-cone", *patterns)
    git("fetch", "-q", "--depth", "1", "--filter=blob:none", "origin", sha)
    git("-c", "advice.detachedHead=false", "checkout", "-q", "FETCH_HEAD")
    return sha


def parse_metadata(path):
    text = path.read_text(encoding="utf-8")
    field = lambda name: re.findall(rf'^{name}: "([^"]*)"', text, flags=re.M)
    fonts = re.findall(r'^\s*filename: "([^"]+)"', text, flags=re.M)
    return {
        "family": (field("name") or [path.parent.name])[0],
        "dir": str(path.parent.relative_to(GF)),
        "license_dir": path.parent.parent.name,
        "category": field("category"),
        "classifications": field("classifications"),
        "subsets": field("subsets"),
        "files": fonts,
    }


def main():
    meta_patterns = ["/*/*/METADATA.pb"]
    sha = sparse_checkout(meta_patterns)
    families = [parse_metadata(p) for p in sorted(GF.glob("*/*/METADATA.pb"))]
    hw_category = [f for f in families if "HANDWRITING" in f["category"]]
    hw_cyr = [f for f in hw_category if "cyrillic" in f["subsets"]]
    hw_class_only = [f for f in families if "HANDWRITING" in f["classifications"]
                     and "HANDWRITING" not in f["category"] and "cyrillic" in f["subsets"]]
    selected = hw_cyr + hw_class_only

    # Завантажуємо файли шрифтів лише відібраних сімейств
    sparse_checkout(meta_patterns + [f"/{f['dir']}/*" for f in selected])
    for f in selected:
        f["paths"] = [str(Path(f["dir"]) / name) for name in f["files"] if (GF / f["dir"] / name).exists()]

    OUT.parent.mkdir(parents=True, exist_ok=True)
    OUT.write_text(json.dumps({
        "google_fonts_commit": sha,
        "criterion": "category = HANDWRITING і subsets ∋ cyrillic; додатково classifications ∋ HANDWRITING",
        "n_families_total": len(families),
        "n_handwriting_category": len(hw_category),
        "n_handwriting_cyrillic": len(hw_cyr),
        "handwriting_cyrillic": [f["family"] for f in hw_cyr],
        "classifications_only_cyrillic": [f["family"] for f in hw_class_only],
        "selected": selected,
    }, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    print(f"google/fonts {sha[:12]}: сімейств {len(families)}, HANDWRITING {len(hw_category)}, "
          f"з них із кирилицею {len(hw_cyr)}")
    print("  " + ", ".join(f["family"] for f in hw_cyr))
    print(f"лише в classifications (з кирилицею): {', '.join(f['family'] for f in hw_class_only) or '—'}")
    print(f"файлів шрифтів: {sum(len(f['paths']) for f in selected)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/fonts/fetch_fonts.py
"""Крок 5: збирає перелік шрифтів для аудиту → results/fonts/font_list.json.

Джерела (усі на зафіксованих версіях, versions/pinned.json):
- Google Fonts: відбір select_fonts.py (рукописні з кирилицею + Shantell Sans), по одному файлу на сімейство;
- Kobzar KS: архів з Wayback Machine, звірка SHA-256 (файл не поширюємо);
- шрифти інструментів синтетики: ukrainian-handwriting-synth (fonts/) і doc_gen (корінь).

Спершу: python3 scripts/fonts/select_fonts.py і python3 scripts/level12/fetch_sources.py
Запуск: python3 scripts/fonts/fetch_fonts.py
"""

import hashlib
import io
import json
import sys
import urllib.request
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "scripts" / "level12"))
from common import REPOS, WORK  # noqa: E402

PINNED = json.loads((ROOT / "versions" / "pinned.json").read_text(encoding="utf-8"))
SELECTION = ROOT / "results" / "fonts" / "selection.json"
OUT = ROOT / "results" / "fonts" / "font_list.json"
FONT_EXT = (".ttf", ".otf", ".TTF", ".OTF")


def pick_file(paths):
    """Один файл на сімейство: змінний шрифт (типовий екземпляр) або Regular, інакше перший."""
    for key in ("[", "-Regular", "Regular"):
        for p in paths:
            if key in Path(p).name and "Italic" not in Path(p).name:
                return p
    return sorted(paths)[0]


def google_fonts():
    sel = json.loads(SELECTION.read_text(encoding="utf-8"))
    rows = []
    for fam in sel["selected"]:
        path = pick_file(fam["paths"])
        rows.append({"name": fam["family"], "source": "google_fonts",
                     "group": "google_fonts" if fam["family"] in sel["handwriting_cyrillic"] else "google_fonts_classification",
                     "path": str((REPOS / "google-fonts" / path).relative_to(WORK)), "license": fam["license_dir"].upper(),
                     "ref": f"google/fonts@{sel['google_fonts_commit'][:12]}:{path}"})
    return rows


def kobzar():
    ext = next(e for e in PINNED["external"] if e["name"] == "Kobzar KS")
    dest = WORK / "kobzar" / ext["file"]
    if not dest.exists():
        data = urllib.request.urlopen(ext["url"], timeout=120).read()
        if hashlib.sha256(data).hexdigest() != ext["sha256_zip"]:
            raise RuntimeError("Kobzar KS: SHA-256 архіву не збігається")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(zipfile.ZipFile(io.BytesIO(data)).read(ext["file"]))
    if hashlib.sha256(dest.read_bytes()).hexdigest() != ext["sha256_file"]:
        raise RuntimeError("Kobzar KS: SHA-256 файлу не збігається")
    return [{"name": "Kobzar KS", "source": "kobzar", "group": "kobzar", "path": str(dest.relative_to(WORK)),
             "license": "Київстар (безоплатна, з обмеженнями)", "ref": f"Wayback 2015-01-23: {ext['file']}"}]


def tool_fonts(repo_slug, subdir, tool):
    root = REPOS / repo_slug / subdir
    sha = next(g["commit"] for g in PINNED["git"] if g["name"] == tool)
    return [{"name": p.stem, "source": tool, "group": tool, "path": str(p.relative_to(WORK)), "license": "не вказано",
             "ref": f"{tool}@{sha[:12]}:{p.relative_to(REPOS / repo_slug)}"}
            for p in sorted(root.iterdir()) if p.suffix in FONT_EXT]


def main():
    fonts = (google_fonts() + kobzar()
             + tool_fonts("ukrainian-handwriting-synth", "fonts", "ukrainian-handwriting-synth")
             + tool_fonts("doc-gen", ".", "doc_gen"))
    for i, f in enumerate(fonts, start=1):
        f["id"] = f"F{i:03d}"
        f["sha256"] = hashlib.sha256((WORK / f["path"]).read_bytes()).hexdigest()
    OUT.write_text(json.dumps({"fonts": fonts}, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    groups = {}
    for f in fonts:
        groups[f["group"]] = groups.get(f["group"], 0) + 1
    print(f"{OUT.relative_to(ROOT)}: {len(fonts)} шрифтів", groups)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/fonts/audit_fonts.py
"""Крок 5: покриття інвентарю й рендеринг для кожного шрифту з results/fonts/font_list.json.

Для кожного символу інвентарю:
- cmap — чи є кодові точки в таблиці cmap (fontTools; для Kobzar KS — HarfBuzz, див. нижче);
- shaping — HarfBuzz: чи потрапляє в результат .notdef (gid 0), чи прикріплено комбінований знак;
- render — Pillow з libraqm (так рендерить ukrainian-handwriting-synth): чи є чорнило на зображенні.
Статус: ok | missing (немає в cmap → .notdef) | empty_glyph (у cmap є, але гліф порожній).
Для комбінованих знаків додатково: чи збігається рендеринг по одному символу (як у doc_gen)
з рендерингом цілого рядка.

Kobzar KS: ліцензія забороняє розбір шрифту, тому для нього таблиці не читаємо fontTools,
а лише рендеримо (HarfBuzz і Pillow) — тобто користуємося шрифтом як будь-яка програма.

Запуск: python3 scripts/fonts/audit_fonts.py
Результати: results/fonts/coverage.json, coverage.csv, summary.md, sheets/*.png,
key_letters.png і key_letters_coding.csv (бланк для кроку 6).
"""

import csv
import json
import os
import sys
import unicodedata
from pathlib import Path

import numpy as np
import uharfbuzz as hb
from fontTools.ttLib import TTFont
from PIL import Image, ImageDraw, ImageFont, features

ROOT = Path(__file__).resolve().parents[2]
WORK = Path(os.environ.get("AUDIT_WORK", Path.home() / ".cache" / "ukr-htg-audit" / "work"))
sys.path.insert(0, str(ROOT / "inventory"))
from load import load_inventory  # noqa: E402

OUT = ROOT / "results" / "fonts"
FONTS = json.loads((OUT / "font_list.json").read_text(encoding="utf-8"))["fonts"]
SIZE = 64
GROUPS = ("alphabet", "apostrophe", "punctuation", "historical")
KEY = ["т", "д", "и", "п", "ш", "г", "б", "в", "з"]
NO_TABLE_READING = {"kobzar"}  # джерела, для яких не читаємо таблиці шрифту (ліцензія)


def hb_font(path):
    blob = hb.Blob.from_file_path(str(path))
    return hb.Font(hb.Face(blob))


def hb_cmap(font, text):
    return [font.get_nominal_glyph(ord(c)) is not None for c in text]


def shape(font, text, feats=None):
    buf = hb.Buffer()
    buf.add_str(text)
    buf.guess_segment_properties()
    hb.shape(font, buf, feats or {})
    return buf.glyph_infos, buf.glyph_positions


def ink(img):
    return int((np.asarray(img) < 128).sum())


def render(pil_font, text, per_char=False):
    """Біле тло, чорний текст; per_char — по одному символу, як у doc_gen."""
    img = Image.new("L", (SIZE * 4 * max(1, len(text)), SIZE * 3), 255)
    draw = ImageDraw.Draw(img)
    if per_char:
        x = SIZE
        for ch in text:
            draw.text((x, SIZE), ch, fill=0, font=pil_font)
            x += draw.textlength(ch, font=pil_font)
    else:
        draw.text((SIZE, SIZE), text, fill=0, font=pil_font)
    return img


def audit_font(f, entries):
    path = WORK / f["path"]  # шляхи у font_list.json — відносно AUDIT_WORK
    font = hb_font(path)
    pil_font = ImageFont.truetype(str(path), SIZE, layout_engine=ImageFont.Layout.RAQM)
    tt_cmap = None
    if f["group"] not in NO_TABLE_READING:
        tt_cmap = TTFont(str(path), lazy=True).getBestCmap() or {}
    rows = {}
    for e in entries:
        text = e["char"]
        in_hb = hb_cmap(font, text)
        in_tt = [ord(c) in tt_cmap for c in text] if tt_cmap is not None else None
        infos, pos = shape(font, text)
        notdef = any(i.codepoint == 0 for i in infos)
        img = render(pil_font, text)
        n_ink = ink(img)
        if not all(in_hb):
            status = "missing"
        elif n_ink == 0:
            status = "empty_glyph"
        else:
            status = "ok"
        row = {"status": status, "cmap": all(in_hb), "notdef": notdef, "ink": n_ink,
               "missing": [f"U+{ord(c):04X}" for c, ok in zip(text, in_hb) if not ok]}
        if in_tt is not None and in_tt != in_hb:
            row["cmap_disagreement"] = {"fontTools": in_tt, "harfbuzz": in_hb}
        if len(text) > 1 and unicodedata.combining(text[-1]) and status == "ok":
            mark = pos[-1]
            row["mark_offset"] = [mark.x_offset, mark.y_offset]
            row["mark_glyphs"] = len(infos)
            whole = np.asarray(img) < 128
            split = np.asarray(render(pil_font, text, per_char=True)) < 128
            row["per_char_differs"] = bool((whole != split).sum() > 0.02 * max(1, whole.sum()))
        rows[e["id"]] = row
    accepted = all(font.get_nominal_glyph(ord(e["char"])) is not None
                   for e in entries if e["group"] == "alphabet")
    return rows, accepted


def sheet(f, entries, rows, label_font):
    cols, cell = 12, 96
    lines = -(-len(entries) // cols)
    img = Image.new("L", (cols * cell, lines * cell + 40), 255)
    draw = ImageDraw.Draw(img)
    draw.text((8, 8), f"{f['id']} {f['name']} — {f['ref']}", fill=0, font=label_font)
    pil_font = ImageFont.truetype(str(WORK / f["path"]), 48, layout_engine=ImageFont.Layout.RAQM)
    for i, e in enumerate(entries):
        x, y = (i % cols) * cell, 40 + (i // cols) * cell
        r = rows[e["id"]]
        draw.rectangle([x, y, x + cell - 1, y + cell - 1], outline=200 if r["status"] == "ok" else 0,
                       width=1 if r["status"] == "ok" else 3)
        draw.text((x + 18, y + 12), e["char"], fill=0, font=pil_font)
        draw.text((x + 4, y + cell - 18), " ".join(e["codepoints"]), fill=110, font=label_font)
    return img


def key_letters(fonts, label_font):
    cell, left = 90, 300
    img = Image.new("L", (left + cell * len(KEY), 50 + cell * len(fonts)), 255)
    draw = ImageDraw.Draw(img)
    for j, ch in enumerate(KEY):
        draw.text((left + j * cell + 35, 15), ch, fill=0, font=ImageFont.truetype(LABEL_CYR, 28))
    for i, f in enumerate(fonts):
        y = 50 + i * cell
        draw.text((8, y + 30), f"{f['id']} {f['name']}"[:34], fill=0, font=label_font)
        pil_font = ImageFont.truetype(str(WORK / f["path"]), 56, layout_engine=ImageFont.Layout.RAQM)
        for j, ch in enumerate(KEY):
            draw.text((left + j * cell + 20, y + 5), ch, fill=0, font=pil_font)
        draw.line([0, y, img.width, y], fill=220)
    return img


def main():
    if not features.check("raqm"):
        raise SystemExit("Pillow без libraqm: результати рендерингу не відповідатимуть інструментам синтетики")
    inv, entries = load_inventory()
    label_font = ImageFont.truetype(LABEL_LATIN, 13)
    (OUT / "sheets").mkdir(parents=True, exist_ok=True)
    results = {}
    for f in FONTS:
        rows, accepted = audit_font(f, entries)
        results[f["id"]] = {"font": f, "accepted_by_ukr_synth": accepted, "entries": rows}
        sheet(f, entries, rows, label_font).save(OUT / "sheets" / f"{f['id']}.png")
        bad = sum(r["status"] != "ok" for r in rows.values())
        print(f"{f['id']} {f['name'][:32]:<32} не ok: {bad:3d}  усі 66 літер: {'так' if accepted else 'ні'}")

    (OUT / "coverage.json").write_text(json.dumps({
        "inventory_version": inv["inventory_version"],
        "render": {"pillow_raqm": features.version("raqm"), "harfbuzz": hb.version_string(), "size_px": SIZE},
        "fonts": results}, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    with open(OUT / "coverage.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["id", "char", "codepoints", "group"] + [f"{r['font']['id']} {r['font']['name']}" for r in results.values()])
        for e in entries:
            w.writerow([e["id"], e["char"], " ".join(e["codepoints"]), e["group"]]
                       + [r["entries"][e["id"]]["status"] for r in results.values()])
    write_summary(results, entries)
    key_letters(FONTS, label_font).save(OUT / "key_letters.png")
    with open(OUT / "key_letters_coding.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["font_id", "font", "letter", "code (відповідає / відмінна / невизначено)", "коментар"])
        for f in FONTS:
            for ch in KEY:
                w.writerow([f["id"], f["name"], ch, "", ""])


def write_summary(results, entries):
    by_group = {g: [e for e in entries if e["group"] == g] for g in GROUPS}
    ukr = [e for e in entries if e["char"] in "ґҐєЄіІїЇ"]
    apo = next(e for e in entries if e["codepoints"] == ["U+02BC"])
    lines = ["# Крок 5: покриття інвентарю шрифтами", "",
             "ok — символ є в cmap і має гліф; missing — немає в cmap (рендериться .notdef); "
             "empty_glyph — у cmap є, але гліф порожній. Повні дані — coverage.json/csv, аркуші — sheets/.", "",
             "| ID | Шрифт | Джерело | " + " | ".join(GROUPS) + " | ґєії | ʼ U+02BC | Не ok |",
             "|---|---|---|" + "---|" * (len(GROUPS) + 3)]
    for r in results.values():
        f, rows = r["font"], r["entries"]
        cells = [f"{sum(rows[e['id']]['status'] == 'ok' for e in by_group[g])}/{len(by_group[g])}" for g in GROUPS]
        ukr_ok = sum(rows[e["id"]]["status"] == "ok" for e in ukr)
        bad = [next(e["char"] for e in entries if e["id"] == i) for i, x in rows.items() if x["status"] != "ok"]
        lines.append(f"| {f['id']} | {f['name']} | {f['group']} | " + " | ".join(cells)
                     + f" | {ukr_ok}/8 | {'так' if rows[apo['id']]['status'] == 'ok' else 'ні'} | "
                     + (" ".join(bad) if len(bad) <= 24 else " ".join(bad[:24]) + f" … (+{len(bad) - 24})") + " |")
    (OUT / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def _label_fonts():
    """Підписи на аркушах — друкованим шрифтом з ukrainian-handwriting-synth (є латиниця й кирилиця)."""
    synth = next((WORK / f["path"]).parents[1] for f in FONTS if f["group"] == "ukrainian-handwriting-synth")
    p = synth / "printed_fonts" / "OpenSans-Regular.ttf"
    return str(p), str(p)


LABEL_LATIN, LABEL_CYR = _label_fonts()

if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/fonts/kobzar_opentype.py
"""Крок 5: чи справді застосовуються OpenType-варіанти Kobzar KS.

Лише рендеринг (HarfBuzz і Pillow з libraqm) — таблиць шрифту fontTools не читаємо, бо
ліцензія Kobzar KS забороняє розбір шрифту. Перевіряємо:
1) які функції GSUB оголошує шрифт (так їх бачить кожен рушій шейпінгу);
2) чи дає типовий шейпінг різні гліфи для тієї самої літери в різних контекстах;
3) що змінюється, якщо вимкнути кожну функцію, і що — якщо ввімкнути необов'язкові (salt, ssXX);
4) чи застосовує їх Pillow типово (як у ukrainian-handwriting-synth) і що лишається при
   рендерингу по одному символу (як у doc_gen).

Запуск: python3 scripts/fonts/kobzar_opentype.py
Результати: results/fonts/kobzar_opentype.json, kobzar_opentype.png
"""

import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import uharfbuzz as hb
from PIL import Image, ImageDraw, ImageFont

ROOT = Path(__file__).resolve().parents[2]
WORK = Path(os.environ.get("AUDIT_WORK", Path.home() / ".cache" / "ukr-htg-audit" / "work"))
sys.path.insert(0, str(ROOT / "inventory"))
from load import load_inventory  # noqa: E402

OUT = ROOT / "results" / "fonts"
FONT = next(f for f in json.loads((OUT / "font_list.json").read_text(encoding="utf-8"))["fonts"]
            if f["group"] == "kobzar")
DEFAULT_ON = {"ccmp", "locl", "rlig", "calt", "clig", "liga", "kern", "mark", "mkmk", "rclt", "curs"}
WORDS = ["Тараса", "Шевченка", "аааа", "ннн", "мама", "оборона", "щастя", "пʼять", "мережа", "Кобзар",
         "садок вишневий коло хати"]


def shape(font, text, feats=None):
    buf = hb.Buffer()
    buf.add_str(text)
    buf.guess_segment_properties()
    hb.shape(font, buf, feats or {})
    return [(i.codepoint, i.cluster) for i in buf.glyph_infos]


def gids(font, text, feats=None):
    return [g for g, _ in shape(font, text, feats)]


def variants_by_char(font, texts, feats=None):
    """Для кожного символу — набір гліфів, якими HarfBuzz його подав у різних контекстах."""
    seen = defaultdict(set)
    for t in texts:
        out = shape(font, t, feats)
        by_cluster = defaultdict(list)
        for g, c in out:
            by_cluster[c].append(g)
        for c, gs in by_cluster.items():
            if t[c].strip():
                seen[t[c]].add(tuple(gs))
    return {ch: len(v) for ch, v in seen.items() if len(v) > 1}


def pil(text, feats=None, per_char=False, size=64):
    font = ImageFont.truetype(str(WORK / FONT["path"]), size, layout_engine=ImageFont.Layout.RAQM)
    img = Image.new("L", (size * len(text) + 2 * size, size * 2), 255)
    draw = ImageDraw.Draw(img)
    if per_char:
        x = size // 2
        for ch in text:
            draw.text((x, size // 3), ch, fill=0, font=font)
            x += draw.textlength(ch, font=font)
    else:
        draw.text((size // 2, size // 3), text, fill=0, font=font, features=feats)
    return img


def feature_tags(face, table):
    """Функції таблиці для всіх скриптів і мов — так їх бачить рушій шейпінгу."""
    tags = set()
    for si, _ in enumerate(face.get_table_script_tags(table)):
        langs = face.get_script_language_tags(table, si)
        for li in [0xFFFF, *range(len(langs))]:
            tags.update(face.get_language_feature_tags(table, si, li))
    return sorted(tags)


def diff(a, b):
    a, b = np.asarray(a) < 128, np.asarray(b) < 128
    return round(float((a != b).sum() / max(1, a.sum())), 3)


def main():
    face = hb.Face(hb.Blob.from_file_path(str(WORK / FONT["path"])))
    font = hb.Font(face)
    gsub, gpos = feature_tags(face, "GSUB"), feature_tags(face, "GPOS")
    inv, entries = load_inventory()
    texts = WORDS + [inv["test_strings"][k] for k in ("pangram", "quick", "punctuation_line")]
    off_all = {t: False for t in gsub}

    per_feature = {}
    for tag in gsub:
        changed = sum(gids(font, t) != gids(font, t, {tag: False}) for t in texts)
        enabled = sum(gids(font, t) != gids(font, t, {tag: True}) for t in texts)
        per_feature[tag] = {"default_on": tag in DEFAULT_ON, "texts_changed_when_off": changed,
                            "texts_changed_when_forced_on": enabled}

    default_vars = variants_by_char(font, texts)
    plain_vars = variants_by_char(font, texts, off_all)
    optional = [t for t in gsub if t not in DEFAULT_ON and per_feature[t]["texts_changed_when_forced_on"]]
    forced_vars = {t: variants_by_char(font, texts, {t: True}) for t in optional}
    forced_all = variants_by_char(font, texts, {t: True for t in ("init", "fina", "salt") if t in gsub})
    # Скільки різних форм шрифт має для кожної літери абетки: ізольована літера
    # з типовими налаштуваннями та з кожною необов'язковою функцією окремо
    settings = [{}] + [{t: True} for t in optional]
    alt_forms = {}
    for e in entries:
        if e["group"] == "alphabet":
            forms = {tuple(gids(font, e["char"], f)) for f in settings}
            alt_forms[e["char"]] = len(forms)
    pil_cmp = {}
    for t in WORDS:
        base = pil(t)
        pil_cmp[t] = {"no_gsub_features": diff(base, pil(t, feats=[f"-{x}" for x in gsub])),
                      "per_char_doc_gen": diff(base, pil(t, per_char=True))}

    result = {
        "font": FONT["ref"], "sha256": FONT["sha256"], "harfbuzz": hb.version_string(),
        "gsub_features": gsub, "gpos_features": gpos, "per_feature": per_feature,
        "chars_with_contextual_variants_default": default_vars,
        "chars_with_variants_all_gsub_off": plain_vars,
        "chars_with_variants_when_forced_on": forced_vars,
        "chars_with_variants_init_fina_salt": forced_all,
        "forms_per_letter_across_features": alt_forms,
        "pillow_pixel_difference": pil_cmp,
        "note": "pixel difference = частка пікселів чорнила, що різняться, відносно типового рендерингу Pillow (libraqm)",
    }
    (OUT / "kobzar_opentype.json").write_text(json.dumps(result, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

    rows = [("типово (Pillow + libraqm, як ukrainian-handwriting-synth)", lambda t: pil(t)),
            ("усі функції GSUB вимкнено", lambda t: pil(t, feats=[f"-{x}" for x in gsub])),
            ("по одному символу (як doc_gen)", lambda t: pil(t, per_char=True))]
    sample = "Тараса Шевченка аааа щастя"
    imgs = [(label, fn(sample)) for label, fn in rows]
    sheet = Image.new("L", (imgs[0][1].width, sum(i.height + 24 for _, i in imgs)), 255)
    d, y = ImageDraw.Draw(sheet), 0
    for label, img in imgs:
        d.text((8, y + 4), label, fill=0)
        sheet.paste(img, (0, y + 24))
        y += img.height + 24
    sheet.save(OUT / "kobzar_opentype.png")

    print("GSUB:", ", ".join(gsub) or "—", "| GPOS:", ", ".join(gpos) or "—")
    for tag, r in per_feature.items():
        print(f"  {tag}: типово {'увімк.' if r['default_on'] else 'вимк.'}; вимкнення змінює {r['texts_changed_when_off']} "
              f"з {len(texts)} рядків; примусове ввімкнення змінює {r['texts_changed_when_forced_on']}")
    print("Літери з кількома варіантами (типово):", default_vars or "—")
    print("…з вимкненим GSUB:", plain_vars or "—")
    for t, v in forced_vars.items():
        print(f"…з увімкненим {t}: {len(v)} літер мають кілька форм", dict(list(v.items())[:12]))
    print(f"…з init+fina+salt: {len(forced_all)} літер мають кілька форм")
    multi = {k: v for k, v in alt_forms.items() if v > 1}
    print(f"Форм на літеру в усіх налаштуваннях: {len(multi)} з {len(alt_forms)} літер мають >1:", multi)
    for t, r in pil_cmp.items():
        print(f"  Pillow «{t}»: без GSUB {r['no_gsub_features']}, по символу {r['per_char_doc_gen']}")


if __name__ == "__main__":
    main()

In [ ]:
# Крок 5 — необов'язковий повторний запуск (результати вже є в results/fonts локально).
# Потрібні репозиторії з fetch_sources.py (ukrainian-handwriting-synth, doc_gen).
# !python3 scripts/fonts/select_fonts.py
# !python3 scripts/fonts/fetch_fonts.py
# !python3 scripts/fonts/audit_fonts.py
# !python3 scripts/fonts/kobzar_opentype.py

## Крок 6. Кодування форм 9 літер

Ручна робота двох незалежних кодувальників на повному наборі: 9 літер (т д и п ш г б в з) × 59 шрифтів = 531 зображення, перемішані й без назв шрифтів (`scripts/coding/make_step6_kit.py` → `coding_kit/step6/`: `кодування_форм.xlsx`, `images/`, `ІНСТРУКЦІЯ.txt`; ключ — `coding_kit/_key/`, кодувальникам не передається). Еталон — чинні прописи для 1 класу, однакове видання для обох. Коди: 1 відповідає / 2 відмінна / 0 невизначено. Надійність — міжекспертна каппа Коена і відсоток збігу на всьому наборі (замість повторного кодування одним експертом). Комплект зібрано локально (потрібні файли шрифтів); у Colab запускати не обов'язково.

## Крок 7. Рівень 3: генерація на GPU

Моделі: українська DiffusionPen, DiffusionPen (IAM), Emuru, Eruku, VATr з розширеним алфавітом. Кожна літера інвентарю (і кожен варіант апострофа) — у 2 словах × 2 стилях (автори 0000 і 0002 зі Space української DiffusionPen). Що не запуститься — у звіті як «не перевірено». Результати — на Google Drive у `MyDrive/ukr-htg-audit/level3`, з відновленням після обриву сесії. Сліпа оцінка — `level3/blind/оцінка_рівень3.xlsx` (коди 1 потрібна літера / 2 інша літера / 0 невпізнавано); ключ `key.csv` відкривається лише після оцінки.

In [ ]:
%%writefile scripts/level3/build_words.py
"""Крок 7: по 2 слова для кожної літери інвентарю й кожного варіанта апострофа → level3/words.json.

Мінускули — у середині або в кінці слова; маюскули — на початку слова (а Ь і Ъ, що не
починають слів, — у слові великими літерами). Історичні слова взято з прикладів хрестоматії
Німчука (сторінки в полі source), а де прикладу немає — з традиційних написань (позначено).
Скрипт перевіряє, що кожне слово містить свій символ.

Запуск: python3 scripts/level3/build_words.py
"""

import json
import sys
import unicodedata
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "inventory"))
from load import load_inventory  # noqa: E402

OUT = ROOT / "level3" / "words.json"
CIRC, ACUTE, GRAVE = chr(0x0302), chr(0x0301), chr(0x0300)

MODERN = {
    "а": ["мама", "вода"], "А": ["Анна", "Астра"], "б": ["хліб", "небо"], "Б": ["Борщ", "Буря"],
    "в": ["вовк", "трава"], "В": ["Весна", "Вітер"], "г": ["нога", "гора"], "Г": ["Голос", "Гарно"],
    "ґ": ["ґанок", "аґрус"], "Ґ": ["Ґанок", "Ґудзик"], "д": ["дуб", "вода"], "Д": ["Дім", "Дорога"],
    "е": ["село", "небо"], "Е": ["Еней", "Етап"], "є": ["моє", "твоє"], "Є": ["Європа", "Єва"],
    "ж": ["ніж", "жито"], "Ж": ["Жито", "Жук"], "з": ["зима", "коза"], "З": ["Зима", "Зірка"],
    "и": ["липа", "син"], "И": ["Ирій", "Индик"], "і": ["літо", "сік"], "І": ["Іван", "Ігор"],
    "ї": ["їжак", "мої"], "Ї": ["Їжак", "Їжа"], "й": ["край", "мій"], "Й": ["Йосип", "Йод"],
    "к": ["рука", "сокіл"], "К": ["Кіт", "Київ"], "л": ["стіл", "липа"], "Л": ["Липа", "Ліс"],
    "м": ["дім", "мама"], "М": ["Мама", "Море"], "н": ["сон", "небо"], "Н": ["Небо", "Ніч"],
    "о": ["око", "сон"], "О": ["Осінь", "Око"], "п": ["липа", "суп"], "П": ["Поле", "Пісня"],
    "р": ["рука", "море"], "Р": ["Ранок", "Річка"], "с": ["сон", "ліс"], "С": ["Сонце", "Село"],
    "т": ["тато", "літо"], "Т": ["Тато", "Тиша"], "у": ["рука", "дуб"], "У": ["Україна", "Урок"],
    "ф": ["фарба", "кафе"], "Ф": ["Фарба", "Форма"], "х": ["хата", "дах"], "Х": ["Хата", "Хліб"],
    "ц": ["вулиця", "цукор"], "Ц": ["Цукор", "Церква"], "ч": ["ніч", "чай"], "Ч": ["Чай", "Час"],
    "ш": ["шапка", "миша"], "Ш": ["Школа", "Шлях"], "щ": ["щастя", "кущ"], "Щ": ["Щастя", "Щука"],
    "ь": ["день", "сіль"], "Ь": ["ДЕНЬ", "СІЛЬ"], "ю": ["пісню", "юшка"], "Ю": ["Юшка", "Юність"],
    "я": ["земля", "ясно"], "Я": ["Ясно", "Яблуко"],
}

# (слово, джерело): с. — сторінка хрестоматії Німчука (2004)
HISTORICAL = {
    "ы": [("лыстъ", "Павловський 1818, с. 40"), ("сыній", "Павловський 1818, с. 40")],
    "Ы": [("СЫНЪ", "капітель; сынъ — пропозиція"), ("СТЫХЪ", "капітель; «Стыхъ», Шевченко 1861, с. 69")],
    "ъ": [("стыхъ", "Шевченко 1861, с. 69"), ("хлѣбъ", "Максимович, с. 89")],
    "Ъ": [("ХЛѢБЪ", "капітель; Максимович, с. 89"), ("СТЫХЪ", "капітель; Шевченко 1861, с. 69")],
    "ѣ": [("снѣгъ", "Максимович, с. 89"), ("свѣти", "Шевченко 1861, с. 69")],
    "Ѣ": [("Ѣда", "традиційне написання"), ("СВѢТЪ", "капітель; Шевченко 1861, с. 69")],
    "ѳ": [("ариѳметика", "Куліш 1857, с. 75"), ("каѳедра", "традиційне написання")],
    "Ѳ": [("Ѳедоръ", "традиційне написання"), ("Ѳома", "традиційне написання")],
    "ѵ": [("мѵро", "традиційне написання"), ("ѵпостась", "традиційне написання")],
    "Ѵ": [("Ѵпостась", "традиційне написання"), ("Ѵподіакон", "традиційне написання")],
    "ё": [("сёго", "Куліш 1857, с. 75"), ("ёго", "Куліш 1857, с. 75 («Ёго»)")],
    "Ё": [("Ёго", "Куліш 1857, с. 75"), ("Ёлка", "традиційне написання")],
    "э": [("эхъ", "традиційне написання"), ("эй", "традиційне написання")],
    "Э": [("Эй", "традиційне написання"), ("Эхъ", "традиційне написання")],
    "j": [("jак", "драгоманівка, с. 161 (ja)"), ("моjа", "драгоманівка, с. 161 (ja)")],
    "J": [("Jак", "драгоманівка, с. 161"), ("Jевген", "драгоманівка, с. 161 (je)")],
    "ѯ": [("Алеѯандръ", "традиційне написання"), ("ѯенія", "традиційне написання")],
    "Ѯ": [("Ѯенія", "традиційне написання"), ("Ѯеноѳонтъ", "традиційне написання")],
    "ѱ": [("ѱалтирь", "традиційне написання"), ("ѱаломъ", "традиційне написання")],
    "Ѱ": [("Ѱалтирь", "традиційне написання"), ("Ѱаломъ", "традиційне написання")],
    "ꙋ": [("мꙋшꙋ", "Гатцук 1861, с. 83"), ("бꙋкварів", "Гатцук 1861, с. 83")],
    "Ꙋ": [("Ꙋкраінска", "Гатцук 1861, с. 83 (назва)"), ("Ꙋсе", "Гатцук 1861, с. 83")],
    "ѧ": [("ѧка", "Гатцук 1861, с. 83"), ("траплѧлосѧ", "Гатцук 1861, с. 83")],
    "Ѧ": [("Ѧк", "Гатцук 1861, с. 83"), ("Ѧка", "Гатцук 1861, с. 83")],
    "о" + CIRC: [("но" + CIRC + "съ", "Максимович, с. 89"), ("мо" + CIRC + "й", "Максимович")],
    "О" + CIRC: [("О" + CIRC + "нъ", "Максимович"), ("НО" + CIRC + "СЪ", "капітель; Максимович, с. 89")],
    "е" + CIRC: [("ме" + CIRC + "дъ", "Максимович, с. 89"), ("пле" + CIRC + "чи", "Максимович")],
    "Е" + CIRC: [("МЕ" + CIRC + "ДЪ", "капітель; Максимович, с. 89"), ("Е" + CIRC + "дъ", "Максимович")],
    "у" + CIRC: [("заму" + CIRC + "жъ", "Максимович, с. 89"), ("ду" + CIRC + "мка", "Максимович")],
    "У" + CIRC: [("ЗАМУ" + CIRC + "ЖЪ", "капітель; Максимович, с. 89"), ("У" + CIRC + "жъ", "Максимович")],
    "и" + CIRC: [("добри" + CIRC, "Максимович, с. 89"), ("женци" + CIRC, "Максимович")],
    "И" + CIRC: [("ДОБРИ" + CIRC, "капітель; Максимович, с. 89"), ("ЖЕНЦИ" + CIRC, "капітель; Максимович")],
    "а" + ACUTE: [("ла" + ACUTE + "вочныця", "Павловський 1818, с. 40"), ("хва" + ACUTE + "рба", "Павловський 1818, с. 40")],
    "А" + ACUTE: [("А" + ACUTE + "збука", "Куліш 1857, с. 75"), ("А" + ACUTE + "нна", "наголос на початку")],
    "а" + GRAVE: [("кра" + GRAVE + "ще", "Шевченко 1861, с. 69"), ("бра" + GRAVE + "томъ", "Шевченко 1861, с. 69")],
    "А" + GRAVE: [("А" + GRAVE + "нна", "наголос на початку"), ("КРА" + GRAVE + "ЩЕ", "капітель; Шевченко 1861, с. 69")],
    "g": [("gвалтъ", "кулішівка (латинська g), Вікіпедія"), ("gудзь", "кулішівка (латинська g), Вікіпедія")],
    "G": [("Gвалтъ", "кулішівка"), ("Gанокъ", "кулішівка")],
}


def main():
    inv, entries = load_inventory()
    apostrophes = [e for e in entries if e["group"] == "apostrophe"]
    units, errors = [], []
    for e in entries:
        ch = e["char"]
        if e["group"] == "alphabet":
            words = [(w, "сучасна лексика") for w in MODERN[ch]]
        elif e["group"] == "historical":
            words = HISTORICAL[ch]
        elif e["group"] == "apostrophe":
            words = [(f"п{ch}ять", "сучасна лексика"), (f"м{ch}ята", "сучасна лексика")]
        else:
            continue  # розділові знаки на рівні 3 не генеруємо
        for i, (w, src) in enumerate(words):
            w = unicodedata.normalize("NFC", w)
            if ch not in w:
                errors.append(f"{e['id']} «{ch}»: слово «{w}» його не містить")
            units.append({"unit": f"{e['id']}_w{i}", "entry_id": e["id"], "char": ch, "group": e["group"],
                          "word": w, "source": src})
    if errors:
        raise SystemExit("\n".join(errors))
    OUT.parent.mkdir(parents=True, exist_ok=True)
    OUT.write_text(json.dumps({"inventory_version": inv["inventory_version"], "units": units},
                              ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    chars = sorted({c for u in units for c in u["word"]})
    print(f"{OUT.relative_to(ROOT)}: {len(units)} слів для {len(units) // 2} символів; "
          f"унікальних символів у словах: {len(chars)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/coding/make_step6_kit.py
"""Крок 6: комплект для незалежного кодування форм 9 літер двома кодувальниками.

Кожна з 9 літер (т д и п ш г б в з) у кожному з 59 шрифтів (results/fonts/font_list.json)
рендериться окремим зображенням (Pillow з libraqm, як в audit_fonts.py). Зображення
перемішано з фіксованим зерном і пронумеровано L001…L531 без назв шрифтів.

Результати:
- coding_kit/step6/images/L*.png — зображення;
- coding_kit/step6/кодування_форм.xlsx — бланк із вбудованими зображеннями і списком кодів;
- coding_kit/_key/step6_key.csv — ключ «номер → шрифт» (кодувальникам не передається).

Запуск: python3 scripts/coding/make_step6_kit.py
Шлях до шрифтів: AUDIT_WORK (типово ~/.cache/ukr-htg-audit/work); шляхи у font_list.json — відносно неї.
"""

import csv
import json
import os
import random
from pathlib import Path

from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation
from PIL import Image, ImageDraw, ImageFont, features

ROOT = Path(__file__).resolve().parents[2]
WORK = Path(os.environ.get("AUDIT_WORK", Path.home() / ".cache" / "ukr-htg-audit" / "work"))
FONTS = json.loads((ROOT / "results" / "fonts" / "font_list.json").read_text(encoding="utf-8"))["fonts"]
KEY_LETTERS = ["т", "д", "и", "п", "ш", "г", "б", "в", "з"]
OUT = ROOT / "coding_kit" / "step6"
KEY = ROOT / "coding_kit" / "_key" / "step6_key.csv"
SEED = 20261027
SIZE, W, H = 96, 200, 160  # розмір кегля і полотна, px


def font_path(f):
    return WORK / f["path"]


def render(path, ch):
    font = ImageFont.truetype(str(path), SIZE, layout_engine=ImageFont.Layout.RAQM)
    img = Image.new("L", (W, H), 255)
    draw = ImageDraw.Draw(img)
    l, t, r, b = draw.textbbox((0, 0), ch, font=font)
    draw.text(((W - (r - l)) / 2 - l, (H - (b - t)) / 2 - t), ch, fill=0, font=font)
    return img


def main():
    if not features.check("raqm"):
        raise SystemExit("Pillow без libraqm")
    items = [(f, ch) for f in FONTS for ch in KEY_LETTERS]
    random.Random(SEED).shuffle(items)
    (OUT / "images").mkdir(parents=True, exist_ok=True)
    KEY.parent.mkdir(parents=True, exist_ok=True)

    wb = Workbook()
    ws = wb.active
    ws.title = "кодування"
    ws.append(["ID", "Зображення", "Літера", "Код: 1 відповідає / 2 відмінна / 0 невизначено", "Коментар"])
    for c in ws[1]:
        c.font = Font(bold=True)
        c.alignment = Alignment(wrap_text=True, vertical="center")
    ws.column_dimensions["A"].width, ws.column_dimensions["B"].width = 8, 30
    ws.column_dimensions["C"].width, ws.column_dimensions["D"].width = 8, 22
    ws.column_dimensions["E"].width = 40
    dv = DataValidation(type="list", formula1='"1,2,0"', allow_blank=True,
                        error="Лише 1, 2 або 0", errorTitle="Невірний код")
    ws.add_data_validation(dv)

    with open(KEY, "w", newline="", encoding="utf-8") as fk:
        key = csv.writer(fk)
        key.writerow(["id", "letter", "font_id", "font", "group"])
        for i, (f, ch) in enumerate(items, start=1):
            iid = f"L{i:03d}"
            png = OUT / "images" / f"{iid}.png"
            render(font_path(f), ch).save(png)
            row = i + 1
            ws.append([iid, None, ch, None, None])
            img = XLImage(str(png))
            img.width, img.height = W * 0.6, H * 0.6
            ws.add_image(img, f"B{row}")
            ws.row_dimensions[row].height = H * 0.6 * 0.75 + 4
            dv.add(f"D{row}")
            for col in "ACD":
                ws[f"{col}{row}"].alignment = Alignment(horizontal="center", vertical="center")
            key.writerow([iid, ch, f["id"], f["name"], f["group"]])
    ws.freeze_panes = "A2"
    wb.save(OUT / "кодування_форм.xlsx")
    print(f"{len(items)} зображень → {OUT}; ключ → {KEY}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/level3/prepare_styles.py
"""Крок 7: два спільні стилі для всіх моделей → level3/styles/.

Джерело — стильові зразки Space української DiffusionPen (UkrHandwritten): автори 0000 і 0002,
по 5 зображень слів. Транскрипції прочитано вручну. Для кожного стилю зберігаємо:
- words/*.png — окремі слова (DiffusionPen, українська DiffusionPen);
- vatr15/*.png — ті самі 5 слів, кожне тричі: VATr бере рівно 15 зразків без повторень;
- line.png + transcription — слова, складені в рядок (Emuru, Eruku).

Запуск: python3 scripts/level3/prepare_styles.py
"""

import json
import shutil
import sys
import urllib.request
from pathlib import Path

from PIL import Image

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "level3" / "styles"
PINNED = json.loads((ROOT / "versions" / "pinned.json").read_text(encoding="utf-8"))
SPACE = next(h for h in PINNED["huggingface"]
             if h["id"] == "kdonitz/diffusionpen-ukrainian" and h["kind"] == "space")

# Транскрипції в порядку імен файлів (переглянуто вручну)
STYLES = {
    "0000": ["був", "Трохи", "але", "й", "принципів"],
    "0002": ["правописам", "у", "за", "потяга", "їх"],
}


def writer_index(writer):
    """Індекс автора в app.py: позиція в упорядкованому списку тек style_refs/ (308 авторів)."""
    dirs = SPACE.get("style_ref_dirs") or sorted(  # скорочений pinned_colab.json має готовий список
        {f["path"].split("/")[1] for f in SPACE["files"] if f["path"].startswith("style_refs/")})
    return dirs.index(writer), len(dirs)


def fetch(path, dest):
    if not dest.exists():
        url = f"https://huggingface.co/spaces/{SPACE['id']}/resolve/{SPACE['sha']}/{path}"
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(urllib.request.urlopen(url, timeout=60).read())


def make_line(images, gap=24, height=64):
    scaled = [im.resize((max(1, im.width * height // im.height), height)) for im in images]
    line = Image.new("L", (sum(i.width for i in scaled) + gap * (len(scaled) + 1), height), 255)
    x = gap
    for im in scaled:
        line.paste(im, (x, 0))
        x += im.width + gap
    return line


def main():
    meta = {"source": f"hf space {SPACE['id']}@{SPACE['sha']}: style_refs/", "styles": {}}
    for writer, words in STYLES.items():
        files = sorted(f["path"] for f in SPACE["files"] if f["path"].startswith(f"style_refs/{writer}/"))
        if len(files) != len(words):
            sys.exit(f"автор {writer}: {len(files)} файлів, а транскрипцій {len(words)}")
        wdir = OUT / writer
        images = []
        for i, path in enumerate(files):
            dest = wdir / "words" / f"{i}.png"
            fetch(path, dest)
            images.append(Image.open(dest).convert("L"))
            for k in range(3):
                (wdir / "vatr15").mkdir(parents=True, exist_ok=True)
                shutil.copyfile(dest, wdir / "vatr15" / f"{i}_{k}.png")
        make_line(images).save(wdir / "line.png")
        idx, n = writer_index(writer)
        meta["styles"][writer] = {"files": files, "words": words, "transcription": " ".join(words),
                                  "ukr_writer_index": idx, "ukr_num_writers": n}
        print(f"стиль {writer}: {len(files)} слів, індекс автора {idx}/{n}, рядок «{' '.join(words)}»")
    (OUT / "styles.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/level3/generators.py
"""Крок 7: завантаження п'яти моделей з відкритими вагами й генерація одного слова.

Кожна функція load_*() повертає (generate, info): generate(word, style) → PIL-зображення,
де style — запис із level3/styles/styles.json (+ шлях до теки стилю). Ваги й код — на
ревізіях із versions/pinned.json. Відхилення від офіційних скриптів позначено в info["deviations"].
"""

import json
import os
import re
import sys
import types
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "scripts" / "level12"))
from common import REPOS, WORK, pinned  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SD = "stable-diffusion-v1-5/stable-diffusion-v1-5"


def rev(model_id):
    return pinned(model_id)["sha"]


def hf(model_id, filename):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(model_id, filename, revision=rev(model_id))


def _sd_parts():
    from diffusers import AutoencoderKL, DDIMScheduler
    vae = AutoencoderKL.from_pretrained(SD, subfolder="vae", revision=rev(SD)).to(DEVICE).eval()
    sched = DDIMScheduler.from_pretrained(SD, subfolder="scheduler", revision=rev(SD))
    return vae, sched


def _canine():
    from transformers import CanineModel, CanineTokenizer
    r = rev("google/canine-c")
    return CanineTokenizer.from_pretrained("google/canine-c", revision=r), CanineModel.from_pretrained("google/canine-c", revision=r)


def _strip_module(sd):
    """Прибирає всі сегменти DataParallel «module.»: у DiffusionPen обгорнуто і модель, і text_encoder."""
    return {re.sub(r"(^|\.)module\.", r"\1", k): v for k, v in sd.items()}


def _in_path(path):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


# --- Українська DiffusionPen ------------------------------------------------------
def load_ukr_diffusionpen():
    space = REPOS / "UkrDiffusionPen-space"
    _in_path(space)
    import torchvision
    from feature_extractor import ImageEncoder
    from generate_sentence import detect_num_classes, generate_single_word, prepare_style_reference_image, strip_dp_prefix
    from unet import UNetModel
    from utils.word_dataset import char_classes

    repo = "kdonitz/diffusionpen-ukrainian"
    state = strip_dp_prefix(torch.load(hf(repo, "ema_ckpt.pt"), map_location="cpu"))
    tokenizer, canine = _canine()
    # Параметри — як у app.py:113–119 (load_models)
    unet = UNetModel(image_size=(64, 256), in_channels=4, model_channels=320, out_channels=4, num_res_blocks=2,
                     attention_resolutions=(1, 1), channel_mult=(1, 1), num_heads=4,
                     num_classes=detect_num_classes(state), context_dim=320, vocab_size=char_classes,
                     text_encoder=canine, args=types.SimpleNamespace(interpolation=False, mix_rate=None))
    unet.load_state_dict(state)
    unet.to(DEVICE).eval()
    vae, sched = _sd_parts()
    style_enc = ImageEncoder(model_name="mobilenetv2_100", num_classes=0, pretrained=False, trainable=False)
    sd = torch.load(hf(repo, "mixed_ukr_mobilenetv2_100.pth"), map_location="cpu")
    md = style_enc.state_dict()
    md.update({k: v for k, v in sd.items() if k in md and md[k].shape == v.shape})
    style_enc.load_state_dict(md)
    style_enc.to(DEVICE).eval()
    tf = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
                                         torchvision.transforms.Normalize((0.5,) * 3, (0.5,) * 3)])

    def style_ref(style):  # app.py:167–180 (_load_writer_style_ref)
        files = sorted((style["dir"] / "words").glob("*.png"))[:5]
        imgs = [tf(prepare_style_reference_image(Image.open(f).convert("RGB"), 64, 256)) for f in files]
        while len(imgs) < 5:
            imgs.append(imgs[0])
        return torch.stack(imgs)

    @torch.no_grad()
    def generate(word, style):
        return generate_single_word(word=word, unet=unet, vae=vae, style_extractor=style_enc, tokenizer=tokenizer,
                                    noise_scheduler=sched, style_ref=style_ref(style), writer_idx=style["ukr_writer_index"],
                                    device=torch.device(DEVICE), cfg_scale=5.0, img_height=64, img_width=256, text_max_len=40)

    return generate, {"weights": f"{repo}@{rev(repo)}", "code": "Space generate_sentence.generate_single_word",
                      "settings": "50 кроків DDIM, cfg_scale 5.0 (як у app.py)", "deviations": []}


# --- DiffusionPen (оригінал, IAM) ------------------------------------------------------
IAM_LABELS = {"0000": 12, "0002": 1}  # номери авторів IAM з прикладів у train.py:302–303


def load_diffusionpen():
    repo_dir = REPOS / "DiffusionPen"
    _in_path(repo_dir)
    import torchvision
    from feature_extractor import ImageEncoder
    from unet import UNetModel
    from utils.auxilary_functions import centered_PIL, image_resize_PIL

    repo = "konnik/DiffusionPen"
    tokenizer, canine = _canine()
    # train.py:556–576 (типові аргументи) і :663 (побудова UNet); vocab_size = 80 (train.py:636)
    unet = UNetModel(image_size=(64, 256), in_channels=4, model_channels=320, out_channels=4, num_res_blocks=1,
                     attention_resolutions=(1, 1), channel_mult=(1, 1), num_heads=4, num_classes=339,
                     context_dim=320, vocab_size=80, text_encoder=canine,
                     args=types.SimpleNamespace(interpolation=False, mix_rate=None))
    unet.load_state_dict(_strip_module(torch.load(hf(repo, "diffusionpen_iam_model_path/models/ema_ckpt.pt"), map_location="cpu")))
    unet.to(DEVICE).eval()
    vae, sched = _sd_parts()
    style_enc = ImageEncoder(model_name="mobilenetv2_100", num_classes=0, pretrained=False, trainable=False)
    sd = _strip_module(torch.load(hf(repo, "style_models/iam_style_diffusionpen.pth"), map_location="cpu"))
    md = style_enc.state_dict()
    md.update({k: v for k, v in sd.items() if k in md and md[k].shape == v.shape})
    style_enc.load_state_dict(md)
    style_enc.to(DEVICE).eval()
    tf = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
                                         torchvision.transforms.Normalize((0.5,) * 3, (0.5,) * 3)])

    def prep(img):  # train.py:357–370
        w, h = img.size
        img = img.resize((int(w * 64 / h), 64))
        w, h = img.size
        if w < 256:
            return ImageOps.pad(img, size=(256, 64), color="white")
        while w > 256:
            img = image_resize_PIL(img, width=w - 20)
            w, h = img.size
        return centered_PIL(img, (64, 256), border_value=255.0)

    @torch.no_grad()
    def generate(word, style):  # train.py:255–430 (Diffusion.sampling) з нашими зразками стилю
        files = sorted((style["dir"] / "words").glob("*.png"))[:5]
        style_images = torch.stack([tf(prep(Image.open(f).convert("RGB"))) for f in files]).to(DEVICE)
        style_features = style_enc(style_images.reshape(-1, 3, 64, 256)).to(DEVICE)
        text = tokenizer(word, padding="max_length", truncation=True, return_tensors="pt", max_length=40).to(DEVICE)
        labels = torch.tensor([IAM_LABELS[style["id"]]], device=DEVICE)
        x = torch.randn((1, 4, 64 // 8, 256 // 8), device=DEVICE)
        sched.set_timesteps(50)
        for time in sched.timesteps:
            t = (torch.ones(1) * time.item()).long().to(DEVICE)
            residual = unet(x, t, text, labels, original_images=style_images, mix_rate=None, style_extractor=style_features)
            x = sched.step(residual, time, x).prev_sample
        image = vae.decode(x / 0.18215).sample
        image = (image / 2 + 0.5).clamp(0, 1)
        return torchvision.transforms.ToPILImage()(image[0].cpu()).convert("L")

    return generate, {"weights": f"{repo}@{rev(repo)}", "code": "train.py Diffusion.sampling (відтворено)",
                      "settings": "50 кроків DDIM, без CFG (як у train.py)",
                      "deviations": ["Зразки стилю — наші 5 слів замість 5 випадкових слів автора IAM (IAM для запуску не потрібен)",
                                     "Мітка автора IAM фіксована: 12 для стилю 0000, 1 для 0002 (train.py:302–303)"]}


# --- Emuru -------------------------------------------------------------------------
def load_emuru():
    from torchvision.transforms import functional as F
    from transformers import AutoModel
    repo = "blowing-up-groundhogs/emuru"
    model = AutoModel.from_pretrained(repo, trust_remote_code=True, revision=rev(repo)).to(DEVICE).eval()

    def style_tensor(style):  # картка моделі: висота 64, нормалізація [-1, 1]
        img = Image.open(style["dir"] / "line.png").convert("RGB")
        img = img.resize((img.width * 64 // img.height, 64))
        return F.normalize(F.to_tensor(img), [0.5], [0.5]).to(DEVICE)

    @torch.no_grad()
    def generate(word, style):
        return model.generate(style_text=style["transcription"], gen_text=word,
                              style_img=style_tensor(style), max_new_tokens=64)

    return generate, {"weights": f"{repo}@{rev(repo)}", "code": "AutoModel (trust_remote_code), model.generate",
                      "settings": "max_new_tokens 64 (картка моделі)",
                      "deviations": ["Стиль — наш рядок із 5 слів, складених з окремих зображень, і його транскрипція"]}


# --- Eruku -------------------------------------------------------------------------
ERUKU_MAX_NEW_TOKENS = 64  # відхилення від типових 512, див. info["deviations"]


def load_eruku():
    from transformers import AutoModel
    repo = "Blowing-Up-Groundhogs/Eruku"
    model = AutoModel.from_pretrained(repo, trust_remote_code=True, revision=rev(repo)).to(DEVICE).eval()

    @torch.no_grad()
    def generate(word, style):
        return model.generate_handwriting(style_image=Image.open(style["dir"] / "line.png").convert("RGB"),
                                          gen_text=word, style_text=style["transcription"], cfg_scale=1.25,
                                          max_new_tokens=ERUKU_MAX_NEW_TOKENS)

    return generate, {"weights": f"{repo}@{rev(repo)}", "code": "AutoModel (trust_remote_code), generate_handwriting",
                      "settings": f"cfg_scale 1.25 (картка моделі), max_new_tokens {ERUKU_MAX_NEW_TOKENS}",
                      "deviations": ["Стиль — наш рядок із 5 слів і його транскрипція",
                                     f"max_new_tokens {ERUKU_MAX_NEW_TOKENS} замість типових 512 (modeling_eruku.py, "
                                     "generate_handwriting): на кириличних словах модель не видає кінця генерації й "
                                     "доходить до межі (4088 px, ~141 с/зобр. на T4); 64 токени = 512 px, як у прикладі "
                                     "картки Emuru (max_new_tokens=64) — досить для одного слова; відрізано лише хвіст "
                                     "після 512 px"]}


# --- VATr з розширеним алфавітом -------------------------------------------------------
VATR_DRIVE_ID = "1cn0t_I3mUDjKSx9Na0dNHB0nPa0fDf1I"  # vatr.pth у теці ваг із README


def load_vatr(extra_chars):
    repo_dir = REPOS / "VATr"
    _in_path(repo_dir)
    cwd = os.getcwd()
    os.chdir(repo_dir)  # VATr відкриває files/… відносними шляхами
    try:
        import generator as vg
        from models.unifont_module import UnifontModule
        if DEVICE == "cpu":  # models/model.py:128 бере типовий device='cuda'
            UnifontModule.__init__.__defaults__ = ("cpu",) + UnifontModule.__init__.__defaults__[1:]
        args = vg.FakeArgs()
        args.device = DEVICE
        base = args.alphabet + args.special_alphabet
        added = "".join(sorted({c for c in extra_chars if c not in base and not c.isspace()}))
        # vocab_size лишається від стандартного алфавіту (generator.py:69): голови OCR і класифікаторів
        # мають ту саму форму, і checkpoint завантажується без змін; нові символи отримують гліфи Unifont
        args.alphabet = args.alphabet + added
        ckpt = WEIGHTS / "vatr.pth"
        if not ckpt.exists():
            import gdown
            ckpt.parent.mkdir(parents=True, exist_ok=True)
            gdown.download(id=VATR_DRIVE_ID, output=str(ckpt), quiet=True)
        writer = vg.VATr_writer(str(ckpt), args)
    finally:
        os.chdir(cwd)

    @torch.no_grad()
    def generate(word, style):
        os.chdir(repo_dir)
        try:
            writer.set_style_folder(str(style["dir"] / "vatr15"), num_examples=15)
            fake = writer.generate([word])[0]
        finally:
            os.chdir(cwd)
        return Image.fromarray(fake)

    import hashlib
    return generate, {"weights": f"Google Drive {VATR_DRIVE_ID} (vatr.pth), sha256 {hashlib.sha256(ckpt.read_bytes()).hexdigest()}",
                      "code": f"aimagelab/VATr@{pinned('VATr')['commit'][:12]} generator.py VATr_writer",
                      "settings": f"алфавіт розширено на {len(added)} символів: {added}",
                      "deviations": ["15 зразків стилю — 5 слів, кожне тричі (FolderDataset бере 15 без повторень)"]}


WEIGHTS = WORK / "weights"

LOADERS = {
    "ukr_diffusionpen": load_ukr_diffusionpen,
    "diffusionpen": load_diffusionpen,
    "emuru": load_emuru,
    "eruku": load_eruku,
    "vatr_extended": load_vatr,
}

In [ ]:
%%writefile scripts/level3/run_level3.py
"""Крок 7: рівень 3 — генерація слів інвентарю моделями з відкритими вагами.

Кожне слово з level3/words.json (по 2 на символ) — у 2 стилях (level3/styles/). Зображення
й журнал пишуться в LEVEL3_OUT (у Colab — тека на Google Drive), тож перерваний запуск
продовжується з місця зупинки. Модель, що не завантажилась, отримує статус «не перевірено».
Статуси зображень у log.jsonl: ok | empty (модель нічого не згенерувала) | error.

Запуск: python3 scripts/level3/run_level3.py <модель> [--limit N]
Моделі: ukr_diffusionpen, diffusionpen, emuru, eruku, vatr_extended, all

Віддалений код Emuru й Eruku (trust_remote_code) не сумісний із transformers 5.x, тому їх
запускаємо з окремих середовищ із версіями з requirements.txt їхніх репозиторіїв
(у Colab — /content/venv_emuru і /content/venv_eruku, див. блокнот).
"""

import argparse
import json
import os
import random
import sys
import time
import traceback
import zlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

# Завантаження через xet іноді обривається («CAS Client Error»); звичайне HTTP — надійніше
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from generators import DEVICE, LOADERS, ROOT  # noqa: E402

OUT = Path(os.environ.get("LEVEL3_OUT", ROOT / "results" / "level3"))


def seed_for(model, unit, style):
    s = zlib.crc32(f"{model}|{unit}|{style}".encode())
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    return s


def run(model, limit=None):
    words = json.loads((ROOT / "level3" / "words.json").read_text(encoding="utf-8"))["units"]
    meta = json.loads((ROOT / "level3" / "styles" / "styles.json").read_text(encoding="utf-8"))
    styles = [{"id": k, "dir": ROOT / "level3" / "styles" / k, **v} for k, v in meta["styles"].items()]
    out = OUT / model
    out.mkdir(parents=True, exist_ok=True)
    log = out / "log.jsonl"
    info_path = out / "model_info.json"
    jobs = [(u, s) for u in words for s in styles][:limit]

    try:
        loader = LOADERS[model]
        extra = "".join(u["word"] for u in words)
        generate, info = loader(extra) if model == "vatr_extended" else loader()
    except Exception as exc:  # noqa: BLE001 — «не перевірено» теж результат
        info_path.write_text(json.dumps({"model": model, "status": "не перевірено",
                                         "error": f"{type(exc).__name__}: {exc}",
                                         "traceback": traceback.format_exc()[-3000:]}, ensure_ascii=False, indent=1))
        print(f"{model}: модель не завантажилась — не перевірено ({type(exc).__name__}: {exc})")
        return
    import diffusers
    import transformers
    info.update(model=model, status="loaded", device=DEVICE, torch=torch.__version__,
                transformers=transformers.__version__, diffusers=diffusers.__version__, python=sys.executable,
                started=datetime.now(timezone.utc).isoformat(timespec="seconds"))
    info_path.write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding="utf-8")

    # Порожній результат не має файлу; з тим самим зерном він повториться, тож не генеруємо його вдруге
    empty = {r["file"] for r in map(json.loads, log.read_text(encoding="utf-8").splitlines())
             if r["status"] == "empty"} if log.exists() else set()
    done = ok = 0
    for unit, style in jobs:
        path = out / f"{unit['unit']}_s{style['id']}.png"
        if path.exists() or path.name in empty:
            continue
        seed = seed_for(model, unit["unit"], style["id"])
        t0 = time.time()
        row = {"unit": unit["unit"], "entry_id": unit["entry_id"], "char": unit["char"], "word": unit["word"],
               "style": style["id"], "seed": seed, "file": path.name}
        try:
            img = generate(unit["word"], style)
            if min(img.size) == 0:  # модель одразу завершила генерацію (Emuru: зріз після зразка стилю порожній)
                row["status"] = "empty"
            else:
                img.save(path)
                row["status"] = "ok"
                ok += 1
        except Exception as exc:  # noqa: BLE001
            row.update(status="error", error=f"{type(exc).__name__}: {exc}"[:300])
        row["seconds"] = round(time.time() - t0, 2)
        with open(log, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        done += 1
        if done % 20 == 0 or done == 1:
            print(f"{model}: {done}/{len(jobs)} ({row['seconds']} с/зобр.)", flush=True)
    print(f"{model}: готово, нових {done}, успішних {ok}; тека {out}")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("model", choices=[*LOADERS, "all"])
    p.add_argument("--limit", type=int, default=None, help="лише перші N зображень (перевірка)")
    a = p.parse_args()
    for m in (LOADERS if a.model == "all" else [a.model]):
        run(m, a.limit)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/level3/make_blind.py
"""Крок 7: набір для сліпої оцінки рівня 3.

Збирає всі успішні зображення з LEVEL3_OUT/<модель>/, перемішує з фіксованим зерном і
копіює під анонімними номерами в LEVEL3_OUT/blind/. Оцінювач бачить лише номер, слово й
цільовий символ; модель і стиль — у ключі key.csv, який відкриваємо після оцінки.

Категорії: потрібна літера / інша літера / невпізнавано.

Запуск: python3 scripts/level3/make_blind.py
"""

import csv
import json
import os
import random
import shutil
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
OUT = Path(os.environ.get("LEVEL3_OUT", ROOT / "results" / "level3"))
BLIND = OUT / "blind"
SEED = 20261024
# Лише основні прогони; controls/ і *_512tokens/ (пробні запуски з іншими налаштуваннями) не входять
MODELS = ["ukr_diffusionpen", "diffusionpen", "emuru", "eruku", "vatr_extended"]


def main():
    rows = []
    for model_dir in (OUT / m for m in MODELS):
        log = model_dir / "log.jsonl"
        if not log.exists():
            continue
        for line in log.read_text(encoding="utf-8").splitlines():
            r = json.loads(line)
            if r["status"] == "ok" and (model_dir / r["file"]).exists():
                rows.append({**r, "model": model_dir.name, "path": model_dir / r["file"]})
    random.Random(SEED).shuffle(rows)
    (BLIND / "images").mkdir(parents=True, exist_ok=True)
    with open(BLIND / "rating.csv", "w", newline="", encoding="utf-8") as fr, \
         open(BLIND / "key.csv", "w", newline="", encoding="utf-8") as fk:
        rate, key = csv.writer(fr), csv.writer(fk)
        rate.writerow(["blind_id", "word", "target", "rating (потрібна / інша / невпізнавано)", "коментар"])
        key.writerow(["blind_id", "model", "unit", "entry_id", "style", "file"])
        for i, r in enumerate(rows, start=1):
            bid = f"B{i:04d}"
            shutil.copyfile(r["path"], BLIND / "images" / f"{bid}.png")
            rate.writerow([bid, r["word"], r["char"], "", ""])
            key.writerow([bid, r["model"], r["unit"], r["entry_id"], r["style"], r["file"]])
    models = sorted({r["model"] for r in rows})
    print(f"{len(rows)} зображень від {len(models)} моделей ({', '.join(models)}) → {BLIND}")
    print("Оцінюйте rating.csv; key.csv не відкривайте до кінця оцінки.")


if __name__ == "__main__":
    main()

In [ ]:
# Крок 7а: слова (214 = 107 символів × 2) і два стилі
!python3 scripts/level3/build_words.py
!python3 scripts/level3/prepare_styles.py

In [ ]:
# Крок 7б: перевірка кожної моделі на 2 зображеннях (ваги завантажуються тут; кілька хвилин на модель).
# Іде у фоні в окремій сесії (клітинка завершується одразу); журнал — $LEVEL3_OUT/smoke.log; перегляд — наступна клітинка.
# Emuru й Eruku — з окремих середовищ (PY, клітинка «Окремі середовища» в Налаштуваннях).
import os, subprocess
MODELS = ["ukr_diffusionpen", "diffusionpen", "emuru", "eruku", "vatr_extended"]
os.makedirs(os.environ["LEVEL3_OUT"], exist_ok=True)
cmd = "; ".join(f'echo "=== {m}"; {PY.get(m, "python3")} scripts/level3/run_level3.py {m} --limit 2 2>&1 '
                f'| grep -v -E "Warning|warn\\(" | tail -n 12' for m in MODELS) + '; echo "=== кінець"'
smoke = subprocess.Popen(["bash", "-c", cmd], stdout=open(os.environ["LEVEL3_OUT"] + "/smoke.log", "w"),
                         stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
print("запущено, pid", smoke.pid)

In [ ]:
# Подивитися перші зображення кожної моделі
import json, os
from pathlib import Path
from IPython.display import display, Image as IPImage
out = Path(os.environ["LEVEL3_OUT"])
for m in ["ukr_diffusionpen", "diffusionpen", "emuru", "eruku", "vatr_extended"]:
    info = out / m / "model_info.json"
    status = json.loads(info.read_text())["status"] if info.exists() else "—"
    print(f"== {m}: {status}")
    for p in sorted((out / m).glob("*.png"))[:2]:
        print(p.name); display(IPImage(str(p)))

In [ ]:
# Діагностика: стан моделей і помилки окремих зображень
import json, os
from pathlib import Path
out = Path(os.environ["LEVEL3_OUT"])
for m in ["ukr_diffusionpen", "diffusionpen", "emuru", "eruku", "vatr_extended"]:
    p = out / m / "model_info.json"
    if not p.exists():
        print("=====", m, "| ще не запускалась"); continue
    i = json.loads(p.read_text())
    print("=====", m, "|", i["status"], "|", i.get("transformers", ""), i.get("diffusers", ""), "|", i.get("error", "")[:400])
    if i["status"] != "loaded":
        print(i.get("traceback", "")[-1500:])
    log = out / m / "log.jsonl"
    for r in (json.loads(l) for l in log.read_text().splitlines()) if log.exists() else []:
        if r["status"] != "ok":
            print("   ", r["unit"], r["style"], r["word"], "→", r.get("error"))

In [ ]:
%%writefile scripts/level3/control_latin.py
"""Крок 7: контроль для Emuru й Eruku — чи справна обгортка, якщо модель провалює кирилицю.

Той самий виклик, що в run_level3.py, у трьох умовах:
- наші стилі (кириличні зразки з кириличною транскрипцією) × латинський і кириличний gen_text;
- для Eruku ще й без транскрипції стилю (style_text необов'язковий, картка моделі);
- зразок стилю з картки Emuru (sample.png, транскрипція 'THE JOLLY IS "U"') × приклад із картки
  ('EMURU'), латиниця й кирилиця — перевіряє середовище й відокремлює вплив кириличного стилю
  від впливу кириличного gen_text.
Уже зроблені випадки (за controls.jsonl) не повторюються.

Запуск: python3 scripts/level3/control_latin.py <emuru|eruku>
Результат: LEVEL3_OUT/controls/<модель>/*.png і controls.jsonl
"""

import json
import os
import shutil
import sys
import time
import zlib
from pathlib import Path

import torch

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from generators import LOADERS, ROOT, hf  # noqa: E402

OUT = Path(os.environ.get("LEVEL3_OUT", ROOT / "results" / "level3")) / "controls"
CARD_STYLE_TEXT = 'THE JOLLY IS "U"'  # картка blowing-up-groundhogs/emuru, приклад використання
CASES = [  # (назва, gen_text, чи передавати транскрипцію стилю, стилі: ours | card)
    ("latin_mama", "mama", True, "ours"),
    ("latin_hello", "hello", True, "ours"),
    ("cyr_mama", "мама", True, "ours"),
    ("card_EMURU", "EMURU", True, "card"),
    ("card_latin_mama", "mama", True, "card"),
    ("card_cyr_mama", "мама", True, "card"),
]
CASES_ERUKU_EXTRA = [
    ("latin_mama_nostyletext", "mama", False, "ours"),
    ("cyr_mama_nostyletext", "мама", False, "ours"),
]


def card_style():
    sdir = OUT / "card_style"
    if not (sdir / "line.png").exists():
        sdir.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(hf("blowing-up-groundhogs/emuru", "sample.png"), sdir / "line.png")
    return {"id": "card", "dir": sdir, "transcription": CARD_STYLE_TEXT}


def main():
    model = sys.argv[1]
    meta = json.loads((ROOT / "level3" / "styles" / "styles.json").read_text(encoding="utf-8"))
    ours = [{"id": k, "dir": ROOT / "level3" / "styles" / k, **v} for k, v in meta["styles"].items()]
    out = OUT / model
    out.mkdir(parents=True, exist_ok=True)
    log = out / "controls.jsonl"
    done = {(r["case"], r["style"]) for r in map(json.loads, log.read_text(encoding="utf-8").splitlines())} \
        if log.exists() else set()
    cases = CASES + (CASES_ERUKU_EXTRA if model == "eruku" else [])
    todo = [(c, s) for c in cases for s in (ours if c[3] == "ours" else [card_style()]) if (c[0], s["id"]) not in done]
    if not todo:
        print(f"{model}: усі контрольні випадки вже зроблено")
        return
    generate, info = LOADERS[model]()
    for (name, text, with_style_text, _), style in todo:
        st = style if with_style_text else {**style, "transcription": ""}
        torch.manual_seed(zlib.crc32(f"{model}|{name}|{style['id']}".encode()))
        t0 = time.time()
        img = generate(text, st)
        row = {"case": name, "text": text, "style": style["id"], "style_text": st["transcription"],
               "size": list(img.size), "seconds": round(time.time() - t0, 1)}
        if min(img.size) > 0:
            img.save(out / f"{name}_s{style['id']}.png")
        with open(log, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(row, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# Контроль Emuru й Eruku (латиниця vs кирилиця) у фоні; журнал — $LEVEL3_OUT/controls.log
import os, subprocess
cmd = "; ".join(f'{PY[m]} scripts/level3/control_latin.py {m} 2>&1 | grep -E "^\\{{|Error|error"' for m in ["emuru", "eruku"])
ctrl = subprocess.Popen(["bash", "-c", cmd], stdout=open(os.environ["LEVEL3_OUT"] + "/controls.log", "w"),
                        stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
print("запущено, pid", ctrl.pid)

In [ ]:
# Крок 7в: повна генерація (5 моделей × 428 зображень) у фоні; продовжується з місця зупинки.
# Окрема сесія: клітинка завершується одразу; журнал — $LEVEL3_OUT/run.log; стан — наступна клітинка.
# Emuru й Eruku — з окремих середовищ (PY); Eruku — max_new_tokens 64 (відхилення, generators.py).
import os, subprocess
MODELS = ["ukr_diffusionpen", "diffusionpen", "vatr_extended", "emuru", "eruku"]
cmd = "; ".join(f'{PY.get(m, "python3")} scripts/level3/run_level3.py {m}' for m in MODELS)
full = subprocess.Popen(["bash", "-c", cmd], stdout=open(os.environ["LEVEL3_OUT"] + "/run.log", "a"),
                        stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
print("запущено, pid", full.pid, MODELS)

In [ ]:
# Стан генерації (запускайте будь-коли). Для кожного зображення береться останній запис журналу.
import json, os, subprocess
from pathlib import Path
out = Path(os.environ["LEVEL3_OUT"])
running = subprocess.run(["pgrep", "-f", "run_level3.py"], capture_output=True, text=True).stdout.split()
print("процес працює" if running else "процес не працює")
for m in ["ukr_diffusionpen", "diffusionpen", "emuru", "eruku", "vatr_extended"]:
    log, info = out / m / "log.jsonl", out / m / "model_info.json"
    last = {}
    for l in (log.read_text().splitlines() if log.exists() else []):
        r = json.loads(l); last[r["file"]] = r["status"]
    try:
        status = json.loads(info.read_text())["status"]
    except (FileNotFoundError, json.JSONDecodeError):
        status = "—"
    n = {s: sum(v == s for v in last.values()) for s in ("ok", "empty", "error")}
    print(f"{m:18s} {status:14s} ok {n['ok']:3d}/428, порожніх {n['empty']}, помилок {n['error']}")
!tail -n 5 "$LEVEL3_OUT/run.log" 2>/dev/null

In [ ]:
# Крок 7г: набір для сліпої оцінки (після завершення генерації).
# Копіювання ~1900 файлів на Drive триває кілька хвилин, тож у фоні; журнал — $LEVEL3_OUT/blind.log
import os, subprocess
b = subprocess.Popen(["python3", "scripts/level3/make_blind.py"], stdout=open(os.environ["LEVEL3_OUT"] + "/blind.log", "w"),
                     stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
print("запущено, pid", b.pid, "— результат: !cat $LEVEL3_OUT/blind.log")

In [ ]:
%%writefile scripts/coding/make_blind_xlsx.py
"""Крок 7: бланк сліпої оцінки рівня 3 у форматі xlsx із вбудованими зображеннями.

Бере теку blind/ з make_blind.py (images/B*.png і rating.csv) і створює поруч
оцінка_рівень3.xlsx: номер, зображення, слово, цільовий символ, код, коментар.
Ключ key.csv не читається і в бланк не потрапляє.

Запуск: python3 scripts/coding/make_blind_xlsx.py <шлях до теки blind>
"""

import csv
import sys
from pathlib import Path

from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation
from PIL import Image

MAX_W, MAX_H = 360, 72  # розмір зображення в бланку, px


def main():
    blind = Path(sys.argv[1])
    rows = list(csv.reader(open(blind / "rating.csv", encoding="utf-8")))[1:]
    wb = Workbook()
    ws = wb.active
    ws.title = "оцінка"
    ws.append(["ID", "Зображення", "Слово", "Цільовий символ",
               "Код: 1 потрібна літера / 2 інша літера / 0 невпізнавано", "Коментар"])
    for c in ws[1]:
        c.font = Font(bold=True)
        c.alignment = Alignment(wrap_text=True, vertical="center")
    for col, width in zip("ABCDEF", (8, 52, 16, 10, 24, 40)):
        ws.column_dimensions[col].width = width
    dv = DataValidation(type="list", formula1='"1,2,0"', allow_blank=True,
                        error="Лише 1, 2 або 0", errorTitle="Невірний код")
    ws.add_data_validation(dv)
    for i, (bid, word, target, *_rest) in enumerate(rows, start=2):
        ws.append([bid, None, word, target, None, None])
        with Image.open(blind / "images" / f"{bid}.png") as im:
            scale = min(MAX_W / im.width, MAX_H / im.height)
        img = XLImage(str(blind / "images" / f"{bid}.png"))
        img.width, img.height = img.width * scale, img.height * scale
        ws.add_image(img, f"B{i}")
        ws.row_dimensions[i].height = MAX_H * 0.75 + 4
        dv.add(f"E{i}")
        for col in "ACDE":
            ws[f"{col}{i}"].alignment = Alignment(horizontal="center", vertical="center")
        ws[f"C{i}"].font = ws[f"D{i}"].font = Font(size=14)
    ws.freeze_panes = "A2"
    wb.save(blind / "оцінка_рівень3.xlsx")
    print(f"{len(rows)} рядків → {blind / 'оцінка_рівень3.xlsx'}")


if __name__ == "__main__":
    main()

In [ ]:
# Крок 7ґ: бланк сліпої оцінки з вбудованими зображеннями (у фоні, кілька хвилин; журнал — $LEVEL3_OUT/blind_xlsx.log)
import os, subprocess
os.makedirs("scripts/coding", exist_ok=True)
x = subprocess.Popen(["python3", "scripts/coding/make_blind_xlsx.py", os.environ["LEVEL3_OUT"] + "/blind"],
                     stdout=open(os.environ["LEVEL3_OUT"] + "/blind_xlsx.log", "w"), stderr=subprocess.STDOUT,
                     stdin=subprocess.DEVNULL, start_new_session=True)
print("запущено, pid", x.pid)